# 02 — research-grade Leakage-Safe Feature Store

This notebook converts the frozen Selection-Sunday snapshots and split contract from notebook `01`
into a **versioned, diagnostic, model-ready feature store** for NCAA men's and women's tournament
forecasting.

It is intentionally broader than any single public competition solution. It preserves the strongest
ideas from public 2026 solutions—seed priors, possession-based efficiency, margin-aware Elo,
opponent adjustment, ranking consensus, matchup symmetry, margin-model support, and gender-aware
routing—while adding stricter temporal provenance, quality sensitivity, hierarchical priors,
multiple independent rating systems, drift analysis, and machine-checkable leakage tests.

## What this notebook builds

1. Compact-result performance, form, volatility, location, rest, and Pythagorean features.
2. Detailed-box-score possession, Four Factors, shooting, rebounding, and style features.
3. Full-data, clean-possession, recent-window, exponentially weighted, and robust variants.
4. Multiple independent team-strength systems:
   - standard and margin-aware Elo;
   - ridge/Massey-style margin ratings;
   - Bradley–Terry strength;
   - Colley ratings;
   - PageRank-style graph strength;
   - opponent-adjusted offensive and defensive efficiency.
5. Strength-of-schedule, quality-win, bad-loss, and conference-context features.
6. Strictly prior-season program and coach tournament-history features.
7. Men's pre-tournament Massey consensus, disagreement, percentile, momentum, and stable-system features.
8. Tournament seed metadata and strictly prequential hierarchical seed-matchup priors.
9. Historical and 2026 matchup feature stores with lower-TeamID orientation and symmetry audits.
10. Missingness, correlation, redundancy, drift, source-provenance, and artifact-fingerprint reports.

## What this notebook deliberately does **not** do

It does not select features, tune hyperparameters, calibrate probabilities, optimize ensemble weights,
or inspect the locked benchmark to choose a model. Those learned decisions belong in notebook `03`
and must occur inside the nested season folds frozen by notebook `01`.

> **Scientific boundary:** every team-season feature for season `Y` is calculated only from regular-
> season information in season `Y` through DayNum 132, plus information from tournament seasons
> strictly earlier than `Y`. Same-season NCAA outcomes are never used as predictors.


## Public-solution synthesis and how this notebook goes further

| Public approach | Retained here | Added here |
|---|---|---|
| 1st place | Separate men's/women's systems, seeds, robust efficiency, quality wins, shallow boosting, calibration | Multiple independent ratings, explicit provenance, clean/full sensitivity, nested feature selection contract, pooled challenger |
| 2nd place | Efficiency, Four Factors, MOV Elo, Massey consensus, SOS, recent form, linear/tree diversity | No fake women's Massey values; gender-specific and pooled stores; hierarchical seed priors; drift diagnostics |
| 4th place | XGBoost + LightGBM, pairwise differences, symmetric augmentation, season grouping | Strict rolling-origin folds, richer graph/statistical ratings, uncertainty and quality propagation |
| 10th/19th places | Point-margin modeling and probability calibration | Stores both win and margin targets under one feature definition; calibrators deferred to nested OOF |
| 21st place | Walk-forward evaluation and gender-specific blending | Partial-pooling architecture and constrained weights selected only from inner OOF predictions |

Public competition work is treated as evidence—not as a license to copy fixed weights, select features
on the locked years, or assume that any feature that worked once will generalize. This notebook creates
a broad **candidate bank**; notebook `03` will force every block to earn inclusion through nested,
season-held-out ablation.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import re
import sys
import warnings
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy import sparse
from scipy.sparse.linalg import spsolve
from scipy.stats import trim_mean
from sklearn.linear_model import LogisticRegression, Ridge

from march_mania.paths import get_project_paths

warnings.filterwarnings("once", category=RuntimeWarning)

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 260)

PATHS = get_project_paths()
ROOT = PATHS.root
INTERIM = PATHS.interim
PROCESSED = PATHS.processed
CONFIG_DIR = ROOT / "configs"
FEATURE_REPORTS = ROOT / "reports" / "feature_engineering"
FIGURE_DIR = ROOT / "reports" / "figures" / "feature_engineering"

for directory in (PROCESSED, CONFIG_DIR, FEATURE_REPORTS, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Interim data:", INTERIM)
print("Processed data:", PROCESSED)
print("Python:", sys.executable)
print("Python version:", platform.python_version())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)

assert "ml-modeling" in str(sys.executable).lower(), (
    "Select the Python (ml-modeling) kernel before continuing."
)


## 1. Load the frozen protocol and all notebook `01` artifacts

The notebook verifies notebook `01` before creating a feature. A changed split contract, incomplete
snapshot coverage, or failed leakage check is a blocking error.


In [ ]:
REQUIRED_INPUTS = {
    "team_game_compact_long": "team_game_compact_long.parquet",
    "detailed_quality": "detailed_regular_game_quality_flags.parquet",
    "base_snapshot": "team_season_snapshot_base.parquet",
    "target_registry": "modeling_target_registry.parquet",
    "outer_folds": "fold_manifest_outer.parquet",
    "inner_folds": "fold_manifest_inner.parquet",
    "locked_folds": "fold_manifest_locked_benchmark.parquet",
    "submission_routing": "submission_routing.parquet",
    "games_compact": "games_compact_canonical.parquet",
    "games_detailed": "games_detailed_team_long.parquet",
    "seeds": "seeds.parquet",
    "team_conferences": "team_conferences.parquet",
    "teams": "teams.parquet",
}

OPTIONAL_INPUTS = {
    "massey": "massey_ordinals_men.parquet",
    "coaches": "coaches_men.parquet",
    "game_cities": "game_cities.parquet",
    "cities": "cities.parquet",
    "conferences": "conferences.parquet",
}

missing = [
    filename for filename in REQUIRED_INPUTS.values()
    if not (INTERIM / filename).exists()
]
assert not missing, (
    "Required notebook 00/01 artifacts are missing. Re-run notebooks 00 and 01. "
    f"Missing: {missing}"
)

frames = {
    name: pd.read_parquet(INTERIM / filename)
    for name, filename in REQUIRED_INPUTS.items()
}
optional_frames = {
    name: pd.read_parquet(INTERIM / filename)
    for name, filename in OPTIONAL_INPUTS.items()
    if (INTERIM / filename).exists()
}

split_path = CONFIG_DIR / "splits.yaml"
readiness_path = ROOT / "reports" / "modeling" / "01_readiness_summary.json"
leakage_report_path = ROOT / "reports" / "modeling" / "leakage_checks.csv"

assert split_path.exists(), "configs/splits.yaml is missing."
assert readiness_path.exists(), "Notebook 01 readiness report is missing."
assert leakage_report_path.exists(), "Notebook 01 leakage report is missing."

SPLITS = yaml.safe_load(split_path.read_text(encoding="utf-8"))
READINESS_01 = json.loads(readiness_path.read_text(encoding="utf-8"))
LEAKAGE_01 = pd.read_csv(leakage_report_path)

assert READINESS_01["status"] == "complete"
assert int(READINESS_01["blocking_leakage_checks_failed"]) == 0
assert float(READINESS_01["target_team_compact_snapshot_coverage"]) == 1.0
assert float(READINESS_01["stage2_both_snapshot_coverage"]) == 1.0
assert LEAKAGE_01["Passed"].astype(bool).all()
assert READINESS_01["split_contract_sha256"] == SPLITS["contract_sha256"]

TARGET_SEASON = int(SPLITS["target_season"])
CUTOFF_DAY = int(SPLITS["feature_cutoff_day"])
DEVELOPMENT_LAST_SEASON = int(SPLITS["development_last_season"])
LOCKED_BENCHMARK = set(map(int, SPLITS["locked_benchmark_seasons"]))

team_games = frames["team_game_compact_long"].copy()
detailed_quality = frames["detailed_quality"].copy()
base_snapshot = frames["base_snapshot"].copy()
target_registry = frames["target_registry"].copy()
outer_folds = frames["outer_folds"].copy()
inner_folds = frames["inner_folds"].copy()
locked_folds = frames["locked_folds"].copy()
submission_routing = frames["submission_routing"].copy()
games_compact = frames["games_compact"].copy()
games_detailed = frames["games_detailed"].copy()
seeds = frames["seeds"].copy()
team_conferences = frames["team_conferences"].copy()
teams = frames["teams"].copy()

inventory = pd.DataFrame(
    [
        {
            "Artifact": name,
            "Rows": len(frame),
            "Columns": frame.shape[1],
            "MemoryMB": round(frame.memory_usage(index=True, deep=True).sum() / 1024**2, 3),
        }
        for name, frame in {**frames, **optional_frames}.items()
    ]
).sort_values("Artifact").reset_index(drop=True)

print(
    json.dumps(
        {
            "split_contract_sha256": SPLITS["contract_sha256"],
            "target_season": TARGET_SEASON,
            "feature_cutoff_day": CUTOFF_DAY,
            "outer_folds": len(outer_folds),
            "inner_folds": len(inner_folds),
            "stage2_matchups": len(submission_routing),
            "optional_sources_found": sorted(optional_frames),
        },
        indent=2,
    )
)
inventory


## 2. Freeze feature definitions and computational switches

This configuration is a **candidate-generation contract**, not a selected final model. It records every
formula and parameterized feature family so later ablation results can be reproduced. Candidate values
are deliberately broad; notebook `03` must select among them using inner folds only.

The expensive rating systems are enabled by default. Set a switch to `False` only for troubleshooting,
then restore it and execute the complete notebook before treating the feature store as final.


In [ ]:
FEATURE_CONFIG: dict[str, Any] = {
    "feature_contract_version": 1,
    "source_split_contract_sha256": SPLITS["contract_sha256"],
    "target_season": TARGET_SEASON,
    "feature_cutoff_day": CUTOFF_DAY,
    "possession_coefficients": [0.475, 0.44],
    "primary_possession_coefficient": 0.475,
    "possession_gap_clean_threshold": float(
        SPLITS["quality_thresholds"]["absolute_possession_gap_warning"]
    ),
    "recent_game_windows": [5, 10, 20],
    "late_day_thresholds": [90, 110, 120],
    "ewm_halflife_games": [5, 10, 20],
    "pythagorean_exponents": [8.0, 10.0, 11.5, 13.91],
    "elo_variants": [
        {
            "name": "standard",
            "k": 20.0,
            "home_advantage": 100.0,
            "carryover": 0.75,
            "mov": "none",
        },
        {
            "name": "mov_log",
            "k": 20.0,
            "home_advantage": 100.0,
            "carryover": 0.75,
            "mov": "log1p",
        },
        {
            "name": "mov_538",
            "k": 20.0,
            "home_advantage": 100.0,
            "carryover": 0.75,
            "mov": "fivethirtyeight",
        },
        {
            "name": "season_reset_mov",
            "k": 24.0,
            "home_advantage": 75.0,
            "carryover": 0.0,
            "mov": "fivethirtyeight",
        },
    ],
    "margin_ridge_alphas": [5.0, 25.0],
    "bradley_terry_c_values": [0.25, 1.0],
    "adjusted_efficiency_ridge_alphas": [10.0, 50.0],
    "seed_prior_pseudocounts": [2.0, 8.0],
    "stable_massey_system_count": 20,
    "stable_massey_selection_ends": DEVELOPMENT_LAST_SEASON,
    "correlation_warning_threshold": 0.985,
    "drift_psi_warning_threshold": 0.20,
    "computational_switches": {
        "run_bradley_terry": True,
        "run_colley": True,
        "run_pagerank": True,
        "run_adjusted_efficiency": True,
        "run_massey_consensus": "massey" in optional_frames,
        "run_coach_history": "coaches" in optional_frames,
    },
}

feature_config_json = json.dumps(
    FEATURE_CONFIG,
    sort_keys=True,
    separators=(",", ":"),
)
FEATURE_CONFIG["feature_contract_sha256"] = hashlib.sha256(
    feature_config_json.encode("utf-8")
).hexdigest()

feature_config_path = CONFIG_DIR / "features.yaml"
feature_config_path.write_text(
    yaml.safe_dump(FEATURE_CONFIG, sort_keys=False),
    encoding="utf-8",
)
(FEATURE_REPORTS / "feature_contract.json").write_text(
    json.dumps(FEATURE_CONFIG, indent=2),
    encoding="utf-8",
)

print("Feature contract:", feature_config_path)
print("Feature contract SHA-256:", FEATURE_CONFIG["feature_contract_sha256"])
print(yaml.safe_dump(FEATURE_CONFIG, sort_keys=False))


## 3. Reusable numerical and audit utilities


In [ ]:
KEY_COLUMNS = ["Gender", "Season", "TeamID"]
MATCHUP_KEY_COLUMNS = ["Gender", "Season", "Team1ID", "Team2ID"]

FEATURE_META: dict[str, dict[str, Any]] = {}


def register_feature(
    name: str,
    *,
    block: str,
    universe: str,
    availability: str = "common",
    source: str,
    temporal_rule: str = "same-season regular season through DayNum 132",
    direction: str = "context_dependent",
    notes: str = "",
) -> None:
    FEATURE_META[name] = {
        "Feature": name,
        "Block": block,
        "Universe": universe,
        "Availability": availability,
        "Source": source,
        "TemporalRule": temporal_rule,
        "Direction": direction,
        "Notes": notes,
    }


def safe_divide(
    numerator: pd.Series | np.ndarray | float,
    denominator: pd.Series | np.ndarray | float,
) -> pd.Series:
    num = pd.Series(numerator, copy=False, dtype="float64")
    den = pd.Series(denominator, copy=False, dtype="float64")
    result = num.div(den.where(den.ne(0)))
    return result.replace([np.inf, -np.inf], np.nan)


def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    mask = values.notna() & weights.notna() & weights.gt(0)
    if not mask.any():
        return float("nan")
    return float(np.average(values.loc[mask], weights=weights.loc[mask]))


def winsorized_mean(series: pd.Series, limits: tuple[float, float] = (0.02, 0.02)) -> float:
    values = pd.to_numeric(series, errors="coerce").dropna().to_numpy(dtype=float)
    if values.size == 0:
        return float("nan")
    lower = np.quantile(values, limits[0])
    upper = np.quantile(values, 1.0 - limits[1])
    return float(np.clip(values, lower, upper).mean())


def median_absolute_deviation(series: pd.Series) -> float:
    values = pd.to_numeric(series, errors="coerce").dropna().to_numpy(dtype=float)
    if values.size == 0:
        return float("nan")
    median = np.median(values)
    return float(np.median(np.abs(values - median)))


def linear_slope(series: pd.Series) -> float:
    values = pd.to_numeric(series, errors="coerce").dropna().to_numpy(dtype=float)
    if values.size < 3 or np.allclose(values, values[0]):
        return 0.0 if values.size else float("nan")
    x = np.arange(values.size, dtype=float)
    return float(np.polyfit(x, values, deg=1)[0])


def ewm_last(series: pd.Series, halflife: float) -> float:
    values = pd.to_numeric(series, errors="coerce")
    if values.notna().sum() == 0:
        return float("nan")
    return float(values.ewm(halflife=halflife, adjust=False, min_periods=1).mean().iloc[-1])


def ensure_unique(frame: pd.DataFrame, keys: Sequence[str], name: str) -> None:
    duplicates = int(frame.duplicated(list(keys)).sum())
    assert duplicates == 0, f"{name} has {duplicates:,} duplicate rows on {list(keys)}."


def prefix_nonkeys(
    frame: pd.DataFrame,
    prefix: str,
    keys: Sequence[str] = KEY_COLUMNS,
) -> pd.DataFrame:
    rename = {column: f"{prefix}{column}" for column in frame.columns if column not in keys}
    return frame.rename(columns=rename)


def merge_feature_block(
    base: pd.DataFrame,
    block: pd.DataFrame,
    *,
    name: str,
    keys: Sequence[str] = KEY_COLUMNS,
) -> pd.DataFrame:
    ensure_unique(block, keys, name)
    overlap = set(base.columns).intersection(block.columns).difference(keys)
    assert not overlap, f"{name} would overwrite columns: {sorted(overlap)[:20]}"
    return base.merge(block, on=list(keys), how="left", validate="one_to_one")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def beta_smoothed_rate(successes: float, trials: float, pseudocount: float) -> float:
    return float((successes + 0.5 * pseudocount) / (trials + pseudocount))


def make_physical_games(canonical: pd.DataFrame) -> pd.DataFrame:
    regular = canonical.loc[
        canonical["GameType"].eq("Regular")
        & canonical["DayNum"].le(CUTOFF_DAY)
    ].copy()
    required = {
        "Gender", "Season", "DayNum", "WTeamID", "LTeamID",
        "WScore", "LScore", "WLoc", "NumOT", "GameKey",
    }
    missing_columns = required.difference(regular.columns)
    assert not missing_columns, f"Canonical games missing: {sorted(missing_columns)}"
    assert regular["GameKey"].is_unique
    return regular.sort_values(
        ["Gender", "Season", "DayNum", "GameKey"]
    ).reset_index(drop=True)


physical_games = make_physical_games(games_compact)
team_games = team_games.sort_values(
    ["Gender", "Season", "DayNum", "GameKey", "TeamID"]
).reset_index(drop=True)
games_detailed = games_detailed.loc[
    games_detailed["GameType"].eq("Regular")
    & games_detailed["DayNum"].le(CUTOFF_DAY)
].sort_values(
    ["Gender", "Season", "DayNum", "GameKey", "TeamID"]
).reset_index(drop=True)

assert physical_games["DayNum"].max() <= CUTOFF_DAY
assert games_detailed["DayNum"].max() <= CUTOFF_DAY
assert physical_games["Season"].max() == TARGET_SEASON
assert base_snapshot[KEY_COLUMNS].duplicated().sum() == 0

print("Physical regular-season games:", f"{len(physical_games):,}")
print("Team-perspective compact rows:", f"{len(team_games):,}")
print("Team-perspective detailed rows:", f"{len(games_detailed):,}")


## 4. Compact-result performance, form, volatility, location, rest, and Pythagorean features

These features use the longest available histories. They are available even when detailed box scores
are missing and therefore form the backbone of compact-universe baselines and the seed-free fallback.


In [ ]:
compact_games = team_games.copy()
compact_games["GameNumber"] = (
    compact_games.groupby(KEY_COLUMNS, observed=True).cumcount() + 1
)
compact_games["RestDays"] = (
    compact_games.groupby(KEY_COLUMNS, observed=True)["DayNum"].diff()
)
compact_games["RestDaysClipped"] = compact_games["RestDays"].clip(lower=0, upper=21)
compact_games["Close3"] = compact_games["Margin"].abs().le(3)
compact_games["Close5"] = compact_games["Margin"].abs().le(5)
compact_games["WonClose3"] = compact_games["Close3"] & compact_games["Win"].eq(1)
compact_games["WonClose5"] = compact_games["Close5"] & compact_games["Win"].eq(1)
compact_games["BlowoutWin10"] = compact_games["Margin"].ge(10)
compact_games["BlowoutWin20"] = compact_games["Margin"].ge(20)
compact_games["BlowoutLoss10"] = compact_games["Margin"].le(-10)
compact_games["BlowoutLoss20"] = compact_games["Margin"].le(-20)
compact_games["UpsideTail"] = compact_games["Margin"].clip(lower=0)
compact_games["DownsideTail"] = (-compact_games["Margin"]).clip(lower=0)

compact_full = (
    compact_games.groupby(KEY_COLUMNS, observed=True)
    .agg(
        compact__games=("GameKey", "nunique"),
        compact__wins=("Win", "sum"),
        compact__win_pct=("Win", "mean"),
        compact__points_for_mean=("TeamScore", "mean"),
        compact__points_against_mean=("OppScore", "mean"),
        compact__margin_mean=("Margin", "mean"),
        compact__margin_median=("Margin", "median"),
        compact__margin_std=("Margin", "std"),
        compact__margin_iqr=("Margin", lambda s: float(s.quantile(0.75) - s.quantile(0.25))),
        compact__margin_mad=("Margin", median_absolute_deviation),
        compact__margin_q10=("Margin", lambda s: float(s.quantile(0.10))),
        compact__margin_q25=("Margin", lambda s: float(s.quantile(0.25))),
        compact__margin_q75=("Margin", lambda s: float(s.quantile(0.75))),
        compact__margin_q90=("Margin", lambda s: float(s.quantile(0.90))),
        compact__margin_trim02=("Margin", lambda s: float(trim_mean(s.to_numpy(dtype=float), 0.02))),
        compact__margin_winsor02=("Margin", winsorized_mean),
        compact__margin_slope=("Margin", linear_slope),
        compact__win_slope=("Win", linear_slope),
        compact__close3_games=("Close3", "sum"),
        compact__close3_wins=("WonClose3", "sum"),
        compact__close5_games=("Close5", "sum"),
        compact__close5_wins=("WonClose5", "sum"),
        compact__blowout_win10_rate=("BlowoutWin10", "mean"),
        compact__blowout_win20_rate=("BlowoutWin20", "mean"),
        compact__blowout_loss10_rate=("BlowoutLoss10", "mean"),
        compact__blowout_loss20_rate=("BlowoutLoss20", "mean"),
        compact__upside_tail_mean=("UpsideTail", "mean"),
        compact__downside_tail_mean=("DownsideTail", "mean"),
        compact__overtime_rate=("NumOT", lambda s: float(s.gt(0).mean())),
        compact__overtime_periods_mean=("NumOT", "mean"),
        compact__rest_days_mean=("RestDaysClipped", "mean"),
        compact__rest_days_std=("RestDaysClipped", "std"),
        compact__first_game_day=("DayNum", "min"),
        compact__last_game_day=("DayNum", "max"),
    )
    .reset_index()
)

compact_full["compact__losses"] = (
    compact_full["compact__games"] - compact_full["compact__wins"]
)
compact_full["compact__close3_win_pct"] = safe_divide(
    compact_full["compact__close3_wins"],
    compact_full["compact__close3_games"],
).to_numpy()
compact_full["compact__close5_win_pct"] = safe_divide(
    compact_full["compact__close5_wins"],
    compact_full["compact__close5_games"],
).to_numpy()
compact_full["compact__days_since_last_game"] = (
    CUTOFF_DAY - compact_full["compact__last_game_day"]
)

# First-half versus second-half change within each team-season.
midpoint = (
    compact_games.groupby(KEY_COLUMNS, observed=True)["GameNumber"]
    .transform("max")
    .add(1)
    .div(2)
)
compact_games["SeasonHalf"] = np.where(
    compact_games["GameNumber"].le(midpoint), "first", "second"
)
half = (
    compact_games.groupby(KEY_COLUMNS + ["SeasonHalf"], observed=True)
    .agg(
        WinPct=("Win", "mean"),
        MarginMean=("Margin", "mean"),
        PointsForMean=("TeamScore", "mean"),
        PointsAgainstMean=("OppScore", "mean"),
    )
    .unstack("SeasonHalf")
)
half.columns = [
    f"compact__half_{metric.lower()}_{period}"
    for metric, period in half.columns
]
half = half.reset_index()
for metric in ("winpct", "marginmean", "pointsformean", "pointsagainstmean"):
    first = f"compact__half_{metric}_first"
    second = f"compact__half_{metric}_second"
    if first in half.columns and second in half.columns:
        half[f"compact__half_delta_{metric}"] = half[second] - half[first]

compact_features = compact_full.merge(
    half,
    on=KEY_COLUMNS,
    how="left",
    validate="one_to_one",
)

# Home, away, and neutral splits.
for location, label in (("H", "home"), ("A", "away"), ("N", "neutral")):
    subset = compact_games.loc[compact_games["TeamLoc"].eq(location)]
    loc = (
        subset.groupby(KEY_COLUMNS, observed=True)
        .agg(
            **{
                f"compact__{label}_games": ("GameKey", "nunique"),
                f"compact__{label}_win_pct": ("Win", "mean"),
                f"compact__{label}_margin_mean": ("Margin", "mean"),
                f"compact__{label}_margin_std": ("Margin", "std"),
            }
        )
        .reset_index()
    )
    compact_features = compact_features.merge(
        loc, on=KEY_COLUMNS, how="left", validate="one_to_one"
    )

# Exact last-N-game windows.
for window in FEATURE_CONFIG["recent_game_windows"]:
    recent = (
        compact_games.groupby(KEY_COLUMNS, observed=True, group_keys=False)
        .tail(int(window))
    )
    recent_agg = (
        recent.groupby(KEY_COLUMNS, observed=True)
        .agg(
            **{
                f"compact__last{window}_games": ("GameKey", "nunique"),
                f"compact__last{window}_win_pct": ("Win", "mean"),
                f"compact__last{window}_margin_mean": ("Margin", "mean"),
                f"compact__last{window}_margin_median": ("Margin", "median"),
                f"compact__last{window}_margin_std": ("Margin", "std"),
                f"compact__last{window}_points_for_mean": ("TeamScore", "mean"),
                f"compact__last{window}_points_against_mean": ("OppScore", "mean"),
            }
        )
        .reset_index()
    )
    compact_features = compact_features.merge(
        recent_agg, on=KEY_COLUMNS, how="left", validate="one_to_one"
    )

# Calendar cutoffs provide a distinct recent-form definition.
for threshold in FEATURE_CONFIG["late_day_thresholds"]:
    late = compact_games.loc[compact_games["DayNum"].ge(int(threshold))]
    late_agg = (
        late.groupby(KEY_COLUMNS, observed=True)
        .agg(
            **{
                f"compact__day{threshold}plus_games": ("GameKey", "nunique"),
                f"compact__day{threshold}plus_win_pct": ("Win", "mean"),
                f"compact__day{threshold}plus_margin_mean": ("Margin", "mean"),
            }
        )
        .reset_index()
    )
    compact_features = compact_features.merge(
        late_agg, on=KEY_COLUMNS, how="left", validate="one_to_one"
    )

# Exponentially weighted end-of-season state.
for halflife in FEATURE_CONFIG["ewm_halflife_games"]:
    ewm = (
        compact_games.groupby(KEY_COLUMNS, observed=True)
        .agg(
            **{
                f"compact__ewm_h{halflife}_win": (
                    "Win", lambda s, h=halflife: ewm_last(s, h)
                ),
                f"compact__ewm_h{halflife}_margin": (
                    "Margin", lambda s, h=halflife: ewm_last(s, h)
                ),
                f"compact__ewm_h{halflife}_points_for": (
                    "TeamScore", lambda s, h=halflife: ewm_last(s, h)
                ),
                f"compact__ewm_h{halflife}_points_against": (
                    "OppScore", lambda s, h=halflife: ewm_last(s, h)
                ),
            }
        )
        .reset_index()
    )
    compact_features = compact_features.merge(
        ewm, on=KEY_COLUMNS, how="left", validate="one_to_one"
    )

# Pythagorean expectation candidates are deterministic, not selected here.
points_for = base_snapshot["PointsForSum"].astype(float)
points_against = base_snapshot["PointsAgainstSum"].astype(float)
pythag = base_snapshot[KEY_COLUMNS].copy()
for exponent in FEATURE_CONFIG["pythagorean_exponents"]:
    tag = str(exponent).replace(".", "p")
    numerator = np.power(points_for.clip(lower=1.0), exponent)
    denominator = numerator + np.power(points_against.clip(lower=1.0), exponent)
    pythag[f"compact__pythag_exp{tag}"] = safe_divide(numerator, denominator).to_numpy()

pythag = pythag.merge(
    compact_features[KEY_COLUMNS + ["compact__win_pct"]],
    on=KEY_COLUMNS,
    how="left",
    validate="one_to_one",
)
for exponent in FEATURE_CONFIG["pythagorean_exponents"]:
    tag = str(exponent).replace(".", "p")
    pythag[f"compact__pythag_residual_exp{tag}"] = (
        pythag["compact__win_pct"] - pythag[f"compact__pythag_exp{tag}"]
    )
pythag = pythag.drop(columns=["compact__win_pct"])

compact_features = compact_features.merge(
    pythag,
    on=KEY_COLUMNS,
    how="left",
    validate="one_to_one",
)
ensure_unique(compact_features, KEY_COLUMNS, "compact features")

for column in compact_features.columns:
    if column not in KEY_COLUMNS:
        register_feature(
            column,
            block="compact_performance",
            universe="compact",
            source="team_game_compact_long + team_season_snapshot_base",
            direction="higher_or_lower_by_definition",
        )

print("Compact feature table:", compact_features.shape)
compact_features.tail()


## 5. Detailed box-score features: possessions, efficiency, Four Factors, shooting, rebounding, and style

The full-season features use **ratio-of-sums** whenever possible; this is more stable than averaging
small-denominator game percentages. Per-game means, medians, trimmed means, clean-possession variants,
recent windows, and EWMs are retained as independent candidates so quality and form assumptions can be
tested rather than hidden.


In [ ]:
detail = games_detailed.copy()

# Attach the physical-game possession-quality flag created by notebook 01.
quality_lookup = detailed_quality[
    ["GameKey", "AbsolutePossessionGap", "PossessionGapFlag"]
].drop_duplicates("GameKey")
detail = detail.drop(
    columns=["AbsolutePossessionGap", "PossessionGapFlag"],
    errors="ignore",
).merge(
    quality_lookup,
    on="GameKey",
    how="left",
    validate="many_to_one",
)
detail["PossessionGapFlag"] = detail["PossessionGapFlag"].fillna(False).astype(bool)

for coefficient in FEATURE_CONFIG["possession_coefficients"]:
    tag = str(coefficient).replace(".", "p")
    team_poss = (
        detail["TeamFGA"]
        - detail["TeamOR"]
        + detail["TeamTO"]
        + coefficient * detail["TeamFTA"]
    )
    opp_poss = (
        detail["OppFGA"]
        - detail["OppOR"]
        + detail["OppTO"]
        + coefficient * detail["OppFTA"]
    )
    detail[f"TeamPoss_{tag}"] = team_poss
    detail[f"OppPoss_{tag}"] = opp_poss
    detail[f"GamePoss_{tag}"] = (team_poss + opp_poss) / 2.0

PRIMARY_POSS_TAG = str(
    FEATURE_CONFIG["primary_possession_coefficient"]
).replace(".", "p")
poss = detail[f"GamePoss_{PRIMARY_POSS_TAG}"]

detail["OffRtg"] = safe_divide(100.0 * detail["TeamScore"], poss).to_numpy()
detail["DefRtg"] = safe_divide(100.0 * detail["OppScore"], poss).to_numpy()
detail["NetRtg"] = detail["OffRtg"] - detail["DefRtg"]
detail["eFG"] = safe_divide(
    detail["TeamFGM"] + 0.5 * detail["TeamFGM3"],
    detail["TeamFGA"],
).to_numpy()
detail["OppeFG"] = safe_divide(
    detail["OppFGM"] + 0.5 * detail["OppFGM3"],
    detail["OppFGA"],
).to_numpy()
detail["TOVPct"] = safe_divide(
    detail["TeamTO"],
    detail["TeamFGA"] + detail["TeamTO"] + 0.475 * detail["TeamFTA"],
).to_numpy()
detail["OppTOVPct"] = safe_divide(
    detail["OppTO"],
    detail["OppFGA"] + detail["OppTO"] + 0.475 * detail["OppFTA"],
).to_numpy()
detail["ORBPct"] = safe_divide(
    detail["TeamOR"],
    detail["TeamOR"] + detail["OppDR"],
).to_numpy()
detail["DRBPct"] = safe_divide(
    detail["TeamDR"],
    detail["TeamDR"] + detail["OppOR"],
).to_numpy()
detail["FTRate"] = safe_divide(detail["TeamFTA"], detail["TeamFGA"]).to_numpy()
detail["OppFTRate"] = safe_divide(detail["OppFTA"], detail["OppFGA"]).to_numpy()
detail["FTMadeRate"] = safe_divide(detail["TeamFTM"], detail["TeamFGA"]).to_numpy()
detail["OppFTMadeRate"] = safe_divide(detail["OppFTM"], detail["OppFGA"]).to_numpy()
detail["FGPct"] = safe_divide(detail["TeamFGM"], detail["TeamFGA"]).to_numpy()
detail["OppFGPct"] = safe_divide(detail["OppFGM"], detail["OppFGA"]).to_numpy()
detail["FG3Pct"] = safe_divide(detail["TeamFGM3"], detail["TeamFGA3"]).to_numpy()
detail["OppFG3Pct"] = safe_divide(detail["OppFGM3"], detail["OppFGA3"]).to_numpy()
detail["FTPct"] = safe_divide(detail["TeamFTM"], detail["TeamFTA"]).to_numpy()
detail["OppFTPct"] = safe_divide(detail["OppFTM"], detail["OppFTA"]).to_numpy()
detail["FG2M"] = detail["TeamFGM"] - detail["TeamFGM3"]
detail["FG2A"] = detail["TeamFGA"] - detail["TeamFGA3"]
detail["OppFG2M"] = detail["OppFGM"] - detail["OppFGM3"]
detail["OppFG2A"] = detail["OppFGA"] - detail["OppFGA3"]
detail["FG2Pct"] = safe_divide(detail["FG2M"], detail["FG2A"]).to_numpy()
detail["OppFG2Pct"] = safe_divide(detail["OppFG2M"], detail["OppFG2A"]).to_numpy()
detail["FG3AttemptRate"] = safe_divide(detail["TeamFGA3"], detail["TeamFGA"]).to_numpy()
detail["OppFG3AttemptRate"] = safe_divide(detail["OppFGA3"], detail["OppFGA"]).to_numpy()
detail["TrueShootingPct"] = safe_divide(
    detail["TeamScore"],
    2.0 * (detail["TeamFGA"] + 0.475 * detail["TeamFTA"]),
).to_numpy()
detail["OppTrueShootingPct"] = safe_divide(
    detail["OppScore"],
    2.0 * (detail["OppFGA"] + 0.475 * detail["OppFTA"]),
).to_numpy()
detail["AssistPerFGM"] = safe_divide(detail["TeamAst"], detail["TeamFGM"]).to_numpy()
detail["OppAssistPerFGM"] = safe_divide(detail["OppAst"], detail["OppFGM"]).to_numpy()
detail["AssistTurnoverRatio"] = safe_divide(detail["TeamAst"], detail["TeamTO"]).to_numpy()
detail["OppAssistTurnoverRatio"] = safe_divide(detail["OppAst"], detail["OppTO"]).to_numpy()
detail["StealPct"] = safe_divide(detail["TeamStl"], poss).to_numpy()
detail["OppStealPct"] = safe_divide(detail["OppStl"], poss).to_numpy()
detail["BlockPct"] = safe_divide(detail["TeamBlk"], detail["OppFG2A"]).to_numpy()
detail["OppBlockPct"] = safe_divide(detail["OppBlk"], detail["FG2A"]).to_numpy()
detail["FoulRate"] = safe_divide(detail["TeamPF"], poss).to_numpy()
detail["OppFoulRate"] = safe_divide(detail["OppPF"], poss).to_numpy()
detail["ReboundShare"] = safe_divide(
    detail["TeamOR"] + detail["TeamDR"],
    detail["TeamOR"] + detail["TeamDR"] + detail["OppOR"] + detail["OppDR"],
).to_numpy()
detail["FourFactorComposite"] = (
    0.40 * detail["eFG"]
    - 0.25 * detail["TOVPct"]
    + 0.20 * detail["ORBPct"]
    + 0.15 * detail["FTMadeRate"]
)
detail["OppFourFactorComposite"] = (
    0.40 * detail["OppeFG"]
    - 0.25 * detail["OppTOVPct"]
    + 0.20 * (1.0 - detail["DRBPct"])
    + 0.15 * detail["OppFTMadeRate"]
)
detail["FourFactorEdge"] = (
    detail["FourFactorComposite"] - detail["OppFourFactorComposite"]
)

for coefficient in FEATURE_CONFIG["possession_coefficients"]:
    tag = str(coefficient).replace(".", "p")
    game_poss = detail[f"GamePoss_{tag}"]
    detail[f"OffRtg_{tag}"] = safe_divide(
        100.0 * detail["TeamScore"], game_poss
    ).to_numpy()
    detail[f"DefRtg_{tag}"] = safe_divide(
        100.0 * detail["OppScore"], game_poss
    ).to_numpy()
    detail[f"NetRtg_{tag}"] = detail[f"OffRtg_{tag}"] - detail[f"DefRtg_{tag}"]

assert detail[["GameKey", "TeamID"]].duplicated().sum() == 0
assert detail.groupby("GameKey", observed=True).size().eq(2).all()
print("Detailed feature-source rows:", detail.shape)
detail.head()


In [ ]:
RAW_SUM_COLUMNS = [
    "TeamScore", "OppScore",
    "TeamFGM", "TeamFGA", "TeamFGM3", "TeamFGA3", "TeamFTM", "TeamFTA",
    "TeamOR", "TeamDR", "TeamAst", "TeamTO", "TeamStl", "TeamBlk", "TeamPF",
    "OppFGM", "OppFGA", "OppFGM3", "OppFGA3", "OppFTM", "OppFTA",
    "OppOR", "OppDR", "OppAst", "OppTO", "OppStl", "OppBlk", "OppPF",
    f"GamePoss_{PRIMARY_POSS_TAG}",
]

PER_GAME_METRICS = [
    "OffRtg", "DefRtg", "NetRtg", "eFG", "OppeFG", "TOVPct", "OppTOVPct",
    "ORBPct", "DRBPct", "FTRate", "OppFTRate", "FTMadeRate", "OppFTMadeRate",
    "FGPct", "OppFGPct", "FG2Pct", "OppFG2Pct", "FG3Pct", "OppFG3Pct",
    "FTPct", "OppFTPct", "FG3AttemptRate", "OppFG3AttemptRate",
    "TrueShootingPct", "OppTrueShootingPct", "AssistPerFGM", "OppAssistPerFGM",
    "AssistTurnoverRatio", "OppAssistTurnoverRatio", "StealPct", "OppStealPct",
    "BlockPct", "OppBlockPct", "FoulRate", "OppFoulRate", "ReboundShare",
    "FourFactorComposite", "OppFourFactorComposite", "FourFactorEdge",
]


def detailed_ratio_of_sums(
    frame: pd.DataFrame,
    *,
    prefix: str,
) -> pd.DataFrame:
    sums = (
        frame.groupby(KEY_COLUMNS, observed=True)[RAW_SUM_COLUMNS]
        .sum(min_count=1)
        .reset_index()
    )
    result = sums[KEY_COLUMNS].copy()
    gp = sums[f"GamePoss_{PRIMARY_POSS_TAG}"]
    result[f"{prefix}games"] = (
        frame.groupby(KEY_COLUMNS, observed=True)["GameKey"]
        .nunique()
        .to_numpy()
    )
    result[f"{prefix}off_rtg_ros"] = safe_divide(
        100.0 * sums["TeamScore"], gp
    ).to_numpy()
    result[f"{prefix}def_rtg_ros"] = safe_divide(
        100.0 * sums["OppScore"], gp
    ).to_numpy()
    result[f"{prefix}net_rtg_ros"] = (
        result[f"{prefix}off_rtg_ros"] - result[f"{prefix}def_rtg_ros"]
    )
    result[f"{prefix}pace_ros"] = safe_divide(
        gp, result[f"{prefix}games"]
    ).to_numpy()
    result[f"{prefix}efg_ros"] = safe_divide(
        sums["TeamFGM"] + 0.5 * sums["TeamFGM3"], sums["TeamFGA"]
    ).to_numpy()
    result[f"{prefix}opp_efg_ros"] = safe_divide(
        sums["OppFGM"] + 0.5 * sums["OppFGM3"], sums["OppFGA"]
    ).to_numpy()
    result[f"{prefix}tov_pct_ros"] = safe_divide(
        sums["TeamTO"],
        sums["TeamFGA"] + sums["TeamTO"] + 0.475 * sums["TeamFTA"],
    ).to_numpy()
    result[f"{prefix}opp_tov_pct_ros"] = safe_divide(
        sums["OppTO"],
        sums["OppFGA"] + sums["OppTO"] + 0.475 * sums["OppFTA"],
    ).to_numpy()
    result[f"{prefix}orb_pct_ros"] = safe_divide(
        sums["TeamOR"], sums["TeamOR"] + sums["OppDR"]
    ).to_numpy()
    result[f"{prefix}drb_pct_ros"] = safe_divide(
        sums["TeamDR"], sums["TeamDR"] + sums["OppOR"]
    ).to_numpy()
    result[f"{prefix}ft_rate_ros"] = safe_divide(
        sums["TeamFTA"], sums["TeamFGA"]
    ).to_numpy()
    result[f"{prefix}opp_ft_rate_ros"] = safe_divide(
        sums["OppFTA"], sums["OppFGA"]
    ).to_numpy()
    result[f"{prefix}ftmade_rate_ros"] = safe_divide(
        sums["TeamFTM"], sums["TeamFGA"]
    ).to_numpy()
    result[f"{prefix}opp_ftmade_rate_ros"] = safe_divide(
        sums["OppFTM"], sums["OppFGA"]
    ).to_numpy()
    result[f"{prefix}fg_pct_ros"] = safe_divide(
        sums["TeamFGM"], sums["TeamFGA"]
    ).to_numpy()
    result[f"{prefix}fg3_pct_ros"] = safe_divide(
        sums["TeamFGM3"], sums["TeamFGA3"]
    ).to_numpy()
    result[f"{prefix}ft_pct_ros"] = safe_divide(
        sums["TeamFTM"], sums["TeamFTA"]
    ).to_numpy()
    result[f"{prefix}fg3_attempt_rate_ros"] = safe_divide(
        sums["TeamFGA3"], sums["TeamFGA"]
    ).to_numpy()
    result[f"{prefix}true_shooting_ros"] = safe_divide(
        sums["TeamScore"],
        2.0 * (sums["TeamFGA"] + 0.475 * sums["TeamFTA"]),
    ).to_numpy()
    result[f"{prefix}assist_to_turnover_ros"] = safe_divide(
        sums["TeamAst"], sums["TeamTO"]
    ).to_numpy()
    result[f"{prefix}steal_pct_ros"] = safe_divide(
        sums["TeamStl"], gp
    ).to_numpy()
    result[f"{prefix}foul_rate_ros"] = safe_divide(
        sums["TeamPF"], gp
    ).to_numpy()
    result[f"{prefix}rebound_share_ros"] = safe_divide(
        sums["TeamOR"] + sums["TeamDR"],
        sums["TeamOR"] + sums["TeamDR"] + sums["OppOR"] + sums["OppDR"],
    ).to_numpy()
    return result


def detailed_distribution_summary(
    frame: pd.DataFrame,
    *,
    prefix: str,
) -> pd.DataFrame:
    named_aggs: dict[str, tuple[str, Any]] = {}
    for metric in PER_GAME_METRICS:
        named_aggs[f"{prefix}{metric.lower()}_mean"] = (metric, "mean")
        named_aggs[f"{prefix}{metric.lower()}_median"] = (metric, "median")
    for metric in ("NetRtg", "OffRtg", "DefRtg", "eFG", "TOVPct", "ORBPct", "FTRate"):
        named_aggs[f"{prefix}{metric.lower()}_trim02"] = (
            metric,
            lambda s: float(trim_mean(s.dropna().to_numpy(dtype=float), 0.02))
            if s.notna().any() else np.nan,
        )
        named_aggs[f"{prefix}{metric.lower()}_std"] = (metric, "std")
    return (
        frame.groupby(KEY_COLUMNS, observed=True)
        .agg(**named_aggs)
        .reset_index()
    )


detail_all_ros = detailed_ratio_of_sums(detail, prefix="detailed__all__")
detail_all_dist = detailed_distribution_summary(detail, prefix="detailed__all__")

clean_detail = detail.loc[~detail["PossessionGapFlag"]].copy()
detail_clean_ros = detailed_ratio_of_sums(clean_detail, prefix="detailed__clean__")
detail_clean_dist = detailed_distribution_summary(clean_detail, prefix="detailed__clean__")

detailed_features = detail_all_ros.merge(
    detail_all_dist, on=KEY_COLUMNS, how="outer", validate="one_to_one"
)
for block in (detail_clean_ros, detail_clean_dist):
    detailed_features = detailed_features.merge(
        block, on=KEY_COLUMNS, how="outer", validate="one_to_one"
    )

# Exact recent windows from detailed games.
for window in FEATURE_CONFIG["recent_game_windows"]:
    recent = (
        detail.groupby(KEY_COLUMNS, observed=True, group_keys=False)
        .tail(int(window))
    )
    recent_summary = (
        recent.groupby(KEY_COLUMNS, observed=True)
        .agg(
            **{
                f"detailed__last{window}__games": ("GameKey", "nunique"),
                f"detailed__last{window}__off_rtg_mean": ("OffRtg", "mean"),
                f"detailed__last{window}__def_rtg_mean": ("DefRtg", "mean"),
                f"detailed__last{window}__net_rtg_mean": ("NetRtg", "mean"),
                f"detailed__last{window}__pace_mean": (
                    f"GamePoss_{PRIMARY_POSS_TAG}", "mean"
                ),
                f"detailed__last{window}__efg_mean": ("eFG", "mean"),
                f"detailed__last{window}__opp_efg_mean": ("OppeFG", "mean"),
                f"detailed__last{window}__tov_pct_mean": ("TOVPct", "mean"),
                f"detailed__last{window}__orb_pct_mean": ("ORBPct", "mean"),
                f"detailed__last{window}__ft_rate_mean": ("FTRate", "mean"),
                f"detailed__last{window}__four_factor_edge_mean": (
                    "FourFactorEdge", "mean"
                ),
            }
        )
        .reset_index()
    )
    detailed_features = detailed_features.merge(
        recent_summary, on=KEY_COLUMNS, how="outer", validate="one_to_one"
    )

# EWM terminal state for core efficiency and style metrics.
for halflife in FEATURE_CONFIG["ewm_halflife_games"]:
    ewm_named: dict[str, tuple[str, Any]] = {}
    for metric in (
        "OffRtg", "DefRtg", "NetRtg", "eFG", "OppeFG", "TOVPct",
        "ORBPct", "FTRate", "FG3AttemptRate", "FourFactorEdge",
    ):
        ewm_named[f"detailed__ewm_h{halflife}__{metric.lower()}"] = (
            metric,
            lambda s, h=halflife: ewm_last(s, h),
        )
    ewm_summary = (
        detail.groupby(KEY_COLUMNS, observed=True)
        .agg(**ewm_named)
        .reset_index()
    )
    detailed_features = detailed_features.merge(
        ewm_summary, on=KEY_COLUMNS, how="outer", validate="one_to_one"
    )

# Location-specific detailed efficiency.
for location, label in (("A", "away"), ("N", "neutral"), ("H", "home")):
    location_frame = detail.loc[detail["TeamLoc"].eq(location)]
    loc_summary = (
        location_frame.groupby(KEY_COLUMNS, observed=True)
        .agg(
            **{
                f"detailed__{label}__games": ("GameKey", "nunique"),
                f"detailed__{label}__off_rtg_mean": ("OffRtg", "mean"),
                f"detailed__{label}__def_rtg_mean": ("DefRtg", "mean"),
                f"detailed__{label}__net_rtg_mean": ("NetRtg", "mean"),
                f"detailed__{label}__pace_mean": (
                    f"GamePoss_{PRIMARY_POSS_TAG}", "mean"
                ),
            }
        )
        .reset_index()
    )
    detailed_features = detailed_features.merge(
        loc_summary, on=KEY_COLUMNS, how="outer", validate="one_to_one"
    )

# Possession-formula sensitivity.
poss_sensitivity = detail[KEY_COLUMNS].drop_duplicates().copy()
for coefficient in FEATURE_CONFIG["possession_coefficients"]:
    tag = str(coefficient).replace(".", "p")
    summary = (
        detail.groupby(KEY_COLUMNS, observed=True)
        .agg(
            **{
                f"detailed__poss{tag}__off_rtg_mean": (f"OffRtg_{tag}", "mean"),
                f"detailed__poss{tag}__def_rtg_mean": (f"DefRtg_{tag}", "mean"),
                f"detailed__poss{tag}__net_rtg_mean": (f"NetRtg_{tag}", "mean"),
            }
        )
        .reset_index()
    )
    poss_sensitivity = poss_sensitivity.merge(
        summary, on=KEY_COLUMNS, how="left", validate="one_to_one"
    )

detailed_features = detailed_features.merge(
    poss_sensitivity,
    on=KEY_COLUMNS,
    how="outer",
    validate="one_to_one",
)

quality_context = base_snapshot[
    [
        "Gender", "Season", "TeamID", "CompactGames", "DetailedGames",
        "MissingDetailedGames", "DetailedCoverageRate", "HasAnyDetailedData",
        "HasCompleteDetailedCoverage", "FlaggedPossessionGames",
        "FlaggedPossessionRate", "RichFeatureCoverageEligible",
    ]
].copy()
quality_context = quality_context.rename(
    columns={
        column: f"quality__{column}"
        for column in quality_context.columns
        if column not in KEY_COLUMNS
    }
)
detailed_features = detailed_features.merge(
    quality_context, on=KEY_COLUMNS, how="outer", validate="one_to_one"
)

ensure_unique(detailed_features, KEY_COLUMNS, "detailed features")
for column in detailed_features.columns:
    if column not in KEY_COLUMNS:
        register_feature(
            column,
            block="detailed_efficiency",
            universe="rich",
            source="games_detailed_team_long + detailed quality flags",
            direction="higher_or_lower_by_definition",
            notes="Includes full, clean-possession, robust, recent, EWM, and formula-sensitivity candidates.",
        )

print("Detailed feature table:", detailed_features.shape)
detailed_features.tail()


## 6. Dynamic and independent team-strength systems

No single rating system is treated as truth. The store includes ratings with different inductive biases:

- **Elo:** chronological, path-dependent, and sensitive to opponent and game order.
- **Margin ridge / Massey-style:** global same-season point-differential strength.
- **Bradley–Terry:** global same-season win-probability strength.
- **Colley:** conservative wins/losses ranking with schedule coupling.
- **PageRank:** graph centrality based on who beat whom.
- **Adjusted offense/defense:** possession-normalized scoring with separate team offense and opponent defense effects.

Every rating uses only regular-season games through DayNum 132. Tournament labels never enter these fits.
The multiple parameterizations are candidates; notebook `03` must choose among them inside inner folds.


In [ ]:
def normalize_rating_columns(
    frame: pd.DataFrame,
    rating_column: str,
    prefix: str,
) -> pd.DataFrame:
    result = frame.copy()
    grouped = result.groupby(["Gender", "Season"], observed=True)[rating_column]
    mean = grouped.transform("mean")
    std = grouped.transform("std").replace(0, np.nan)
    result[f"{prefix}z"] = (result[rating_column] - mean) / std
    result[f"{prefix}percentile"] = grouped.rank(pct=True, method="average")
    return result


def compute_elo_variant(
    games: pd.DataFrame,
    *,
    name: str,
    k: float,
    home_advantage: float,
    carryover: float,
    mov: str,
    initial: float = 1500.0,
) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    for gender, gender_games in games.groupby("Gender", sort=True, observed=True):
        ratings: dict[int, float] = {}
        for season, season_games in gender_games.groupby("Season", sort=True, observed=True):
            season_games = season_games.sort_values(["DayNum", "GameKey"])
            active_teams = sorted(
                set(season_games["WTeamID"]).union(season_games["LTeamID"])
            )
            # Regress returning teams toward the common prior before the season.
            ratings = {
                team_id: initial + carryover * (rating - initial)
                for team_id, rating in ratings.items()
            }
            preseason = {
                int(team_id): float(ratings.get(int(team_id), initial))
                for team_id in active_teams
            }
            for row in season_games.itertuples(index=False):
                winner = int(row.WTeamID)
                loser = int(row.LTeamID)
                w_rating = float(ratings.get(winner, initial))
                l_rating = float(ratings.get(loser, initial))
                location_adjustment = (
                    home_advantage if row.WLoc == "H"
                    else -home_advantage if row.WLoc == "A"
                    else 0.0
                )
                w_adjusted = w_rating + location_adjustment
                expected = 1.0 / (1.0 + 10.0 ** ((l_rating - w_adjusted) / 400.0))
                margin = max(float(row.WScore - row.LScore), 1.0)
                if mov == "none":
                    multiplier = 1.0
                elif mov == "log1p":
                    multiplier = math.log1p(margin)
                elif mov == "fivethirtyeight":
                    rating_gap = abs(w_adjusted - l_rating)
                    multiplier = math.log1p(margin) * (2.2 / (0.001 * rating_gap + 2.2))
                else:
                    raise ValueError(f"Unknown Elo MOV rule: {mov}")
                change = k * multiplier * (1.0 - expected)
                ratings[winner] = w_rating + change
                ratings[loser] = l_rating - change

            for team_id in active_teams:
                final_rating = float(ratings.get(int(team_id), initial))
                records.append(
                    {
                        "Gender": gender,
                        "Season": int(season),
                        "TeamID": int(team_id),
                        f"rating__elo_{name}": final_rating,
                        f"rating__elo_{name}_preseason": preseason[int(team_id)],
                        f"rating__elo_{name}_delta": final_rating - preseason[int(team_id)],
                    }
                )

    result = pd.DataFrame(records)
    result = normalize_rating_columns(
        result,
        f"rating__elo_{name}",
        f"rating__elo_{name}_",
    )
    ensure_unique(result, KEY_COLUMNS, f"Elo {name}")
    return result


elo_blocks: list[pd.DataFrame] = []
for variant in FEATURE_CONFIG["elo_variants"]:
    print("Computing Elo:", variant["name"])
    elo_blocks.append(compute_elo_variant(physical_games, **variant))

rating_features = elo_blocks[0]
for block in elo_blocks[1:]:
    rating_features = rating_features.merge(
        block, on=KEY_COLUMNS, how="outer", validate="one_to_one"
    )

for column in rating_features.columns:
    if column not in KEY_COLUMNS:
        register_feature(
            column,
            block="dynamic_ratings",
            universe="compact",
            source="chronological regular-season compact games",
            temporal_rule="chronological through DayNum 132; cross-season carry uses prior seasons only",
            direction="higher_is_stronger",
        )

print("Elo feature table:", rating_features.shape)
rating_features.tail()


In [ ]:
def pairwise_design(
    frame: pd.DataFrame,
    teams_in_season: list[int],
    *,
    include_location: bool = True,
) -> tuple[sparse.csr_matrix, dict[int, int]]:
    team_index = {int(team_id): index for index, team_id in enumerate(teams_in_season)}
    n_rows = len(frame)
    n_team_columns = len(teams_in_season)
    extra = 1 if include_location else 0

    row_indices = np.repeat(np.arange(n_rows), 2)
    col_indices = np.concatenate(
        [
            frame["TeamID"].map(team_index).to_numpy(),
            frame["OppTeamID"].map(team_index).to_numpy(),
        ]
    )
    data = np.concatenate([np.ones(n_rows), -np.ones(n_rows)])
    matrix = sparse.coo_matrix(
        (data, (np.tile(np.arange(n_rows), 2), col_indices)),
        shape=(n_rows, n_team_columns + extra),
    ).tocsr()

    if include_location:
        home = frame["TeamLoc"].map({"H": 1.0, "A": -1.0, "N": 0.0}).fillna(0.0)
        matrix = matrix.tolil()
        matrix[:, -1] = home.to_numpy().reshape(-1, 1)
        matrix = matrix.tocsr()
    return matrix, team_index


def compute_margin_ridge_ratings(
    frame: pd.DataFrame,
    alpha: float,
) -> pd.DataFrame:
    records: list[pd.DataFrame] = []
    for (gender, season), group in frame.groupby(
        ["Gender", "Season"], sort=True, observed=True
    ):
        teams_in_season = sorted(
            set(group["TeamID"]).union(group["OppTeamID"])
        )
        matrix, team_index = pairwise_design(
            group, teams_in_season, include_location=True
        )
        model = Ridge(
            alpha=float(alpha),
            fit_intercept=False,
            solver="lsqr",
            tol=1e-7,
        )
        model.fit(matrix, group["Margin"].to_numpy(dtype=float))
        inverse = {index: team_id for team_id, index in team_index.items()}
        tag = str(alpha).replace(".", "p")
        block = pd.DataFrame(
            {
                "Gender": gender,
                "Season": int(season),
                "TeamID": [inverse[index] for index in range(len(teams_in_season))],
                f"rating__margin_ridge_a{tag}": model.coef_[: len(teams_in_season)],
                f"rating__margin_ridge_a{tag}_home_effect": float(model.coef_[-1]),
            }
        )
        block = normalize_rating_columns(
            block,
            f"rating__margin_ridge_a{tag}",
            f"rating__margin_ridge_a{tag}_",
        )
        records.append(block)
    return pd.concat(records, ignore_index=True)


for alpha in FEATURE_CONFIG["margin_ridge_alphas"]:
    print("Computing margin-ridge rating, alpha:", alpha)
    block = compute_margin_ridge_ratings(team_games, float(alpha))
    rating_features = rating_features.merge(
        block, on=KEY_COLUMNS, how="outer", validate="one_to_one"
    )

if FEATURE_CONFIG["computational_switches"]["run_bradley_terry"]:
    def compute_bradley_terry(
        frame: pd.DataFrame,
        c_value: float,
    ) -> pd.DataFrame:
        records: list[pd.DataFrame] = []
        for (gender, season), group in frame.groupby(
            ["Gender", "Season"], sort=True, observed=True
        ):
            teams_in_season = sorted(
                set(group["TeamID"]).union(group["OppTeamID"])
            )
            matrix, team_index = pairwise_design(
                group, teams_in_season, include_location=True
            )
            model = LogisticRegression(
                C=float(c_value),
                fit_intercept=False,
                solver="lbfgs",
                max_iter=1000,
                tol=1e-7,
            )
            model.fit(matrix, group["Win"].astype(int).to_numpy())
            inverse = {index: team_id for team_id, index in team_index.items()}
            tag = str(c_value).replace(".", "p")
            block = pd.DataFrame(
                {
                    "Gender": gender,
                    "Season": int(season),
                    "TeamID": [
                        inverse[index] for index in range(len(teams_in_season))
                    ],
                    f"rating__bt_c{tag}": model.coef_[0, : len(teams_in_season)],
                    f"rating__bt_c{tag}_home_effect": float(model.coef_[0, -1]),
                }
            )
            block = normalize_rating_columns(
                block,
                f"rating__bt_c{tag}",
                f"rating__bt_c{tag}_",
            )
            records.append(block)
        return pd.concat(records, ignore_index=True)

    for c_value in FEATURE_CONFIG["bradley_terry_c_values"]:
        print("Computing Bradley-Terry rating, C:", c_value)
        block = compute_bradley_terry(team_games, float(c_value))
        rating_features = rating_features.merge(
            block, on=KEY_COLUMNS, how="outer", validate="one_to_one"
        )

ensure_unique(rating_features, KEY_COLUMNS, "rating features after ridge/BT")
print("Rating feature table after ridge and Bradley-Terry:", rating_features.shape)


In [ ]:
if FEATURE_CONFIG["computational_switches"]["run_colley"]:
    def compute_colley(games: pd.DataFrame) -> pd.DataFrame:
        records: list[pd.DataFrame] = []
        for (gender, season), group in games.groupby(
            ["Gender", "Season"], sort=True, observed=True
        ):
            teams_in_season = sorted(
                set(group["WTeamID"]).union(group["LTeamID"])
            )
            team_index = {
                int(team_id): index for index, team_id in enumerate(teams_in_season)
            }
            n = len(teams_in_season)
            meetings: defaultdict[tuple[int, int], int] = defaultdict(int)
            wins = np.zeros(n, dtype=float)
            losses = np.zeros(n, dtype=float)

            for row in group.itertuples(index=False):
                w = team_index[int(row.WTeamID)]
                l = team_index[int(row.LTeamID)]
                wins[w] += 1.0
                losses[l] += 1.0
                meetings[(w, l)] += 1
                meetings[(l, w)] += 1

            games_played = wins + losses
            rows = list(range(n))
            cols = list(range(n))
            values = list(2.0 + games_played)
            for (i, j), count in meetings.items():
                rows.append(i)
                cols.append(j)
                values.append(-float(count))

            matrix = sparse.coo_matrix(
                (values, (rows, cols)), shape=(n, n)
            ).tocsr()
            rhs = 1.0 + 0.5 * (wins - losses)
            ratings = spsolve(matrix, rhs)
            block = pd.DataFrame(
                {
                    "Gender": gender,
                    "Season": int(season),
                    "TeamID": teams_in_season,
                    "rating__colley": ratings,
                }
            )
            block = normalize_rating_columns(
                block, "rating__colley", "rating__colley_"
            )
            records.append(block)
        return pd.concat(records, ignore_index=True)

    print("Computing Colley ratings")
    colley = compute_colley(physical_games)
    rating_features = rating_features.merge(
        colley, on=KEY_COLUMNS, how="outer", validate="one_to_one"
    )


if FEATURE_CONFIG["computational_switches"]["run_pagerank"]:
    def compute_pagerank(
        games: pd.DataFrame,
        *,
        margin_weighted: bool,
        damping: float = 0.85,
        tolerance: float = 1e-12,
        max_iter: int = 500,
    ) -> pd.DataFrame:
        records: list[pd.DataFrame] = []
        suffix = "margin" if margin_weighted else "binary"
        for (gender, season), group in games.groupby(
            ["Gender", "Season"], sort=True, observed=True
        ):
            teams_in_season = sorted(
                set(group["WTeamID"]).union(group["LTeamID"])
            )
            team_index = {
                int(team_id): index for index, team_id in enumerate(teams_in_season)
            }
            n = len(teams_in_season)
            source: list[int] = []
            destination: list[int] = []
            weights: list[float] = []
            for row in group.itertuples(index=False):
                source.append(team_index[int(row.LTeamID)])
                destination.append(team_index[int(row.WTeamID)])
                margin = max(float(row.WScore - row.LScore), 1.0)
                weights.append(math.log1p(margin) if margin_weighted else 1.0)

            adjacency = sparse.coo_matrix(
                (weights, (source, destination)), shape=(n, n)
            ).tocsr()
            out_strength = np.asarray(adjacency.sum(axis=1)).ravel()
            inv_out = np.divide(
                1.0,
                out_strength,
                out=np.zeros_like(out_strength),
                where=out_strength > 0,
            )
            transition = sparse.diags(inv_out) @ adjacency
            scores = np.full(n, 1.0 / n)
            teleport = np.full(n, (1.0 - damping) / n)

            for _ in range(max_iter):
                dangling_mass = scores[out_strength == 0].sum()
                updated = (
                    teleport
                    + damping * (transition.T @ scores)
                    + damping * dangling_mass / n
                )
                if np.abs(updated - scores).sum() < tolerance:
                    scores = updated
                    break
                scores = updated

            block = pd.DataFrame(
                {
                    "Gender": gender,
                    "Season": int(season),
                    "TeamID": teams_in_season,
                    f"rating__pagerank_{suffix}": scores,
                }
            )
            block = normalize_rating_columns(
                block,
                f"rating__pagerank_{suffix}",
                f"rating__pagerank_{suffix}_",
            )
            records.append(block)
        return pd.concat(records, ignore_index=True)

    for margin_weighted in (False, True):
        label = "margin" if margin_weighted else "binary"
        print("Computing PageRank:", label)
        pagerank = compute_pagerank(
            physical_games, margin_weighted=margin_weighted
        )
        rating_features = rating_features.merge(
            pagerank, on=KEY_COLUMNS, how="outer", validate="one_to_one"
        )

ensure_unique(rating_features, KEY_COLUMNS, "all compact ratings")
for column in rating_features.columns:
    if column not in KEY_COLUMNS and column not in FEATURE_META:
        register_feature(
            column,
            block="global_strength_ratings",
            universe="compact",
            source="same-season regular-season game graph",
            direction="higher_is_stronger",
        )

print("All compact rating features:", rating_features.shape)
rating_features.tail()


## 7. Opponent-adjusted offense/defense and schedule-accomplishment features

The raw efficiency summaries answer “what happened?” These models answer “against whom?” by
estimating team offense and opponent defense simultaneously. Separate ridge penalties are preserved as
candidates. Schedule and quality-win features are then computed from independent same-season ratings.


In [ ]:
def offense_defense_design(
    frame: pd.DataFrame,
    teams_in_season: list[int],
) -> tuple[sparse.csr_matrix, dict[int, int]]:
    team_index = {int(team_id): index for index, team_id in enumerate(teams_in_season)}
    n_rows = len(frame)
    n_teams = len(teams_in_season)
    row_ids = np.arange(n_rows)
    offense_cols = frame["TeamID"].map(team_index).to_numpy()
    defense_cols = frame["OppTeamID"].map(team_index).to_numpy() + n_teams
    home_col = np.full(n_rows, 2 * n_teams)

    rows = np.concatenate([row_ids, row_ids, row_ids])
    cols = np.concatenate([offense_cols, defense_cols, home_col])
    home_values = (
        frame["TeamLoc"].map({"H": 1.0, "A": -1.0, "N": 0.0})
        .fillna(0.0)
        .to_numpy(dtype=float)
    )
    values = np.concatenate(
        [np.ones(n_rows), np.ones(n_rows), home_values]
    )
    matrix = sparse.coo_matrix(
        (values, (rows, cols)),
        shape=(n_rows, 2 * n_teams + 1),
    ).tocsr()
    return matrix, team_index


def compute_adjusted_efficiency(
    frame: pd.DataFrame,
    alpha: float,
) -> pd.DataFrame:
    records: list[pd.DataFrame] = []
    tag = str(alpha).replace(".", "p")
    for (gender, season), group in frame.groupby(
        ["Gender", "Season"], sort=True, observed=True
    ):
        usable = group.loc[
            group["OffRtg"].notna()
            & group["TeamID"].notna()
            & group["OppTeamID"].notna()
        ].copy()
        if usable.empty:
            continue
        teams_in_season = sorted(
            set(usable["TeamID"]).union(usable["OppTeamID"])
        )
        matrix, team_index = offense_defense_design(usable, teams_in_season)
        model = Ridge(
            alpha=float(alpha),
            fit_intercept=True,
            solver="lsqr",
            tol=1e-7,
        )
        model.fit(matrix, usable["OffRtg"].to_numpy(dtype=float))
        n_teams = len(teams_in_season)
        inverse = {index: team_id for team_id, index in team_index.items()}
        offense = model.intercept_ + model.coef_[:n_teams]
        defense_allowed = model.intercept_ + model.coef_[n_teams:2 * n_teams]
        block = pd.DataFrame(
            {
                "Gender": gender,
                "Season": int(season),
                "TeamID": [inverse[index] for index in range(n_teams)],
                f"adj__off_rtg_a{tag}": offense,
                f"adj__def_rtg_a{tag}": defense_allowed,
                f"adj__net_rtg_a{tag}": offense - defense_allowed,
                f"adj__home_effect_a{tag}": float(model.coef_[-1]),
                f"adj__league_off_rtg_a{tag}": float(model.intercept_),
            }
        )
        block = normalize_rating_columns(
            block,
            f"adj__net_rtg_a{tag}",
            f"adj__net_rtg_a{tag}_",
        )
        records.append(block)
    return pd.concat(records, ignore_index=True)


adjusted_features: pd.DataFrame | None = None
if FEATURE_CONFIG["computational_switches"]["run_adjusted_efficiency"]:
    for alpha in FEATURE_CONFIG["adjusted_efficiency_ridge_alphas"]:
        print("Computing opponent-adjusted efficiency, alpha:", alpha)
        block = compute_adjusted_efficiency(detail, float(alpha))
        adjusted_features = (
            block if adjusted_features is None
            else adjusted_features.merge(
                block, on=KEY_COLUMNS, how="outer", validate="one_to_one"
            )
        )

if adjusted_features is None:
    adjusted_features = base_snapshot[KEY_COLUMNS].copy()

ensure_unique(adjusted_features, KEY_COLUMNS, "adjusted efficiency")
for column in adjusted_features.columns:
    if column not in KEY_COLUMNS:
        register_feature(
            column,
            block="opponent_adjusted_efficiency",
            universe="rich",
            source="same-season regular-season detailed game model",
            direction="higher_is_stronger_except_defense_allowed",
            notes="Ridge estimates team offense and opponent defense simultaneously.",
        )

print("Adjusted efficiency table:", adjusted_features.shape)
adjusted_features.tail()


In [ ]:
# Primary independent strength references used to describe schedule difficulty.
SCHEDULE_RATING = "rating__margin_ridge_a25p0"
if SCHEDULE_RATING not in rating_features.columns:
    candidate_ridge = [
        column for column in rating_features
        if column.startswith("rating__margin_ridge_a")
        and not column.endswith(("_z", "_percentile", "_home_effect"))
    ]
    assert candidate_ridge, "No margin-ridge rating found for schedule features."
    SCHEDULE_RATING = sorted(candidate_ridge)[-1]

schedule_source = team_games.merge(
    rating_features[
        ["Gender", "Season", "TeamID", SCHEDULE_RATING]
    ].rename(
        columns={
            "TeamID": "OppTeamID",
            SCHEDULE_RATING: "OpponentStrength",
        }
    ),
    on=["Gender", "Season", "OppTeamID"],
    how="left",
    validate="many_to_one",
)

schedule_source["OpponentStrengthPct"] = (
    schedule_source.groupby(["Gender", "Season"], observed=True)["OpponentStrength"]
    .rank(pct=True, method="average")
)
schedule_source["QualityWin"] = (
    schedule_source["Win"].eq(1)
    & schedule_source["OpponentStrengthPct"].ge(0.75)
)
schedule_source["EliteWin"] = (
    schedule_source["Win"].eq(1)
    & schedule_source["OpponentStrengthPct"].ge(0.90)
)
schedule_source["BadLoss"] = (
    schedule_source["Win"].eq(0)
    & schedule_source["OpponentStrengthPct"].le(0.25)
)
schedule_source["QualityWinPoints"] = np.where(
    schedule_source["Win"].eq(1),
    schedule_source["OpponentStrengthPct"].fillna(0.5),
    0.0,
)
schedule_source["BadLossPenalty"] = np.where(
    schedule_source["Win"].eq(0),
    1.0 - schedule_source["OpponentStrengthPct"].fillna(0.5),
    0.0,
)

schedule_features = (
    schedule_source.groupby(KEY_COLUMNS, observed=True)
    .agg(
        schedule__opponent_strength_mean=("OpponentStrength", "mean"),
        schedule__opponent_strength_median=("OpponentStrength", "median"),
        schedule__opponent_strength_std=("OpponentStrength", "std"),
        schedule__opponent_strength_pct_mean=("OpponentStrengthPct", "mean"),
        schedule__quality_wins=("QualityWin", "sum"),
        schedule__elite_wins=("EliteWin", "sum"),
        schedule__bad_losses=("BadLoss", "sum"),
        schedule__quality_win_points_sum=("QualityWinPoints", "sum"),
        schedule__quality_win_points_mean=("QualityWinPoints", "mean"),
        schedule__bad_loss_penalty_sum=("BadLossPenalty", "sum"),
        schedule__top_quartile_games=(
            "OpponentStrengthPct", lambda s: int(s.ge(0.75).sum())
        ),
        schedule__top_decile_games=(
            "OpponentStrengthPct", lambda s: int(s.ge(0.90).sum())
        ),
    )
    .reset_index()
)

# RPI-style opponent winning percentages, excluding head-to-head meetings.
team_totals = (
    team_games.groupby(KEY_COLUMNS, observed=True)
    .agg(TotalWins=("Win", "sum"), TotalGames=("GameKey", "nunique"))
    .reset_index()
)
head_to_head = (
    team_games.groupby(KEY_COLUMNS + ["OppTeamID"], observed=True)
    .agg(H2HWins=("Win", "sum"), H2HGames=("GameKey", "nunique"))
    .reset_index()
)
opp_totals = team_totals.rename(
    columns={
        "TeamID": "OppTeamID",
        "TotalWins": "OppTotalWins",
        "TotalGames": "OppTotalGames",
    }
)
rpi_games = head_to_head.merge(
    opp_totals,
    on=["Gender", "Season", "OppTeamID"],
    how="left",
    validate="many_to_one",
)
rpi_games["OpponentWinPctExcludingTeam"] = safe_divide(
    rpi_games["OppTotalWins"] - (
        rpi_games["H2HGames"] - rpi_games["H2HWins"]
    ),
    rpi_games["OppTotalGames"] - rpi_games["H2HGames"],
).to_numpy()

owp = (
    rpi_games.groupby(KEY_COLUMNS, observed=True)
    .agg(schedule__owp=("OpponentWinPctExcludingTeam", "mean"))
    .reset_index()
)
rpi_base = team_totals.merge(
    owp, on=KEY_COLUMNS, how="left", validate="one_to_one"
)
rpi_base["schedule__wp"] = safe_divide(
    rpi_base["TotalWins"], rpi_base["TotalGames"]
).to_numpy()

opp_owp = owp.rename(
    columns={"TeamID": "OppTeamID", "schedule__owp": "OpponentOWP"}
)
oowp_source = head_to_head.merge(
    opp_owp,
    on=["Gender", "Season", "OppTeamID"],
    how="left",
    validate="many_to_one",
)
oowp = (
    oowp_source.groupby(KEY_COLUMNS, observed=True)
    .agg(schedule__oowp=("OpponentOWP", "mean"))
    .reset_index()
)
rpi_base = rpi_base.merge(
    oowp, on=KEY_COLUMNS, how="left", validate="one_to_one"
)
rpi_base["schedule__rpi"] = (
    0.25 * rpi_base["schedule__wp"]
    + 0.50 * rpi_base["schedule__owp"]
    + 0.25 * rpi_base["schedule__oowp"]
)
rpi_base = rpi_base.drop(columns=["TotalWins", "TotalGames"])

schedule_features = schedule_features.merge(
    rpi_base, on=KEY_COLUMNS, how="outer", validate="one_to_one"
)
ensure_unique(schedule_features, KEY_COLUMNS, "schedule features")

for column in schedule_features.columns:
    if column not in KEY_COLUMNS:
        register_feature(
            column,
            block="schedule_and_accomplishment",
            universe="compact",
            source="regular-season opponent graph + independent margin rating",
            direction="higher_is_stronger_except_bad_loss_penalty",
        )

print("Schedule/accomplishment feature table:", schedule_features.shape)
schedule_features.tail()


In [ ]:
# Conference context is derived from team ratings, not same-season NCAA outcomes.
conference_keys = team_conferences[
    ["Gender", "Season", "TeamID", "ConfAbbrev"]
].drop_duplicates(KEY_COLUMNS)
conference_strength_source = conference_keys.merge(
    rating_features[
        ["Gender", "Season", "TeamID", SCHEDULE_RATING]
    ],
    on=KEY_COLUMNS,
    how="left",
    validate="one_to_one",
)

conference_features = (
    conference_strength_source.groupby(
        ["Gender", "Season", "ConfAbbrev"], observed=True
    )
    .agg(
        conference__team_count=("TeamID", "nunique"),
        conference__strength_mean=(SCHEDULE_RATING, "mean"),
        conference__strength_median=(SCHEDULE_RATING, "median"),
        conference__strength_std=(SCHEDULE_RATING, "std"),
        conference__strength_max=(SCHEDULE_RATING, "max"),
    )
    .reset_index()
)
conference_features["conference__strength_percentile"] = (
    conference_features.groupby(["Gender", "Season"], observed=True)[
        "conference__strength_mean"
    ].rank(pct=True, method="average")
)

conference_team_features = conference_keys.merge(
    conference_features,
    on=["Gender", "Season", "ConfAbbrev"],
    how="left",
    validate="many_to_one",
).drop(columns=["ConfAbbrev"])

# Optional conference-tournament result features from raw files.
conference_tourney_parts: list[pd.DataFrame] = []
for gender, filename in (
    ("M", "MConferenceTourneyGames.csv"),
    ("W", "WConferenceTourneyGames.csv"),
):
    raw_path = PATHS.raw / filename
    if raw_path.exists():
        raw = pd.read_csv(raw_path)
        raw["Gender"] = gender
        conference_tourney_parts.append(raw)

if conference_tourney_parts:
    conf_tourney = pd.concat(conference_tourney_parts, ignore_index=True)
    winner = conf_tourney[
        ["Gender", "Season", "DayNum", "WTeamID", "ConfAbbrev"]
    ].rename(columns={"WTeamID": "TeamID"})
    winner["ConfTourneyWin"] = 1
    winner["ConfTourneyGame"] = 1
    loser = conf_tourney[
        ["Gender", "Season", "DayNum", "LTeamID", "ConfAbbrev"]
    ].rename(columns={"LTeamID": "TeamID"})
    loser["ConfTourneyWin"] = 0
    loser["ConfTourneyGame"] = 1
    conf_team_long = pd.concat([winner, loser], ignore_index=True)

    conf_tourney_features = (
        conf_team_long.groupby(KEY_COLUMNS, observed=True)
        .agg(
            conference__tourney_games=("ConfTourneyGame", "sum"),
            conference__tourney_wins=("ConfTourneyWin", "sum"),
            conference__tourney_last_day=("DayNum", "max"),
        )
        .reset_index()
    )
    conf_tourney_features["conference__tourney_win_pct"] = safe_divide(
        conf_tourney_features["conference__tourney_wins"],
        conf_tourney_features["conference__tourney_games"],
    ).to_numpy()

    # A conference champion is the winner in the final game for each conference-season.
    final_rows = conf_tourney.loc[
        conf_tourney["DayNum"].eq(
            conf_tourney.groupby(
                ["Gender", "Season", "ConfAbbrev"], observed=True
            )["DayNum"].transform("max")
        )
    ]
    champions = final_rows[
        ["Gender", "Season", "WTeamID"]
    ].rename(columns={"WTeamID": "TeamID"}).drop_duplicates(KEY_COLUMNS)
    champions["conference__tourney_champion"] = 1
    conf_tourney_features = conf_tourney_features.merge(
        champions,
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one",
    )
    conf_tourney_features["conference__tourney_champion"] = (
        conf_tourney_features["conference__tourney_champion"]
        .fillna(0)
        .astype("int8")
    )
    conference_team_features = conference_team_features.merge(
        conf_tourney_features,
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one",
    )

ensure_unique(conference_team_features, KEY_COLUMNS, "conference features")
for column in conference_team_features.columns:
    if column not in KEY_COLUMNS:
        register_feature(
            column,
            block="conference_context",
            universe="compact",
            source="team conferences + regular-season ratings + conference tournament table",
            direction="higher_is_generally_stronger",
        )

print("Conference feature table:", conference_team_features.shape)
conference_team_features.tail()


## 8. Strictly prior-season program and coach tournament history

These features use NCAA tournament outcomes only from seasons **strictly earlier** than the prediction
season. They are produced by an explicit prequential loop and include provenance columns recording the
latest tournament season allowed to contribute. Current-season tournament performance can never enter.


In [ ]:
def tournament_team_outcomes(targets: pd.DataFrame) -> pd.DataFrame:
    t1 = targets[
        ["Gender", "Season", "DayNum", "Team1ID", "Team1Win", "TargetKey"]
    ].rename(columns={"Team1ID": "TeamID", "Team1Win": "Win"})
    t2 = targets[
        ["Gender", "Season", "DayNum", "Team2ID", "Team1Win", "TargetKey"]
    ].rename(columns={"Team2ID": "TeamID"})
    t2["Win"] = 1 - t2["Team1Win"]
    t2 = t2.drop(columns=["Team1Win"])
    long = pd.concat([t1, t2], ignore_index=True)

    final_day = targets.groupby(
        ["Gender", "Season"], observed=True
    )["DayNum"].transform("max")
    finals = targets.loc[targets["DayNum"].eq(final_day)].copy()
    finals["ChampionTeamID"] = np.where(
        finals["Team1Win"].eq(1), finals["Team1ID"], finals["Team2ID"]
    )
    final_teams = pd.concat(
        [
            finals[["Gender", "Season", "Team1ID"]].rename(
                columns={"Team1ID": "TeamID"}
            ),
            finals[["Gender", "Season", "Team2ID"]].rename(
                columns={"Team2ID": "TeamID"}
            ),
        ],
        ignore_index=True,
    ).drop_duplicates(KEY_COLUMNS)
    final_teams["ReachedFinal"] = 1
    champions = finals[
        ["Gender", "Season", "ChampionTeamID"]
    ].rename(columns={"ChampionTeamID": "TeamID"}).drop_duplicates(KEY_COLUMNS)
    champions["Champion"] = 1

    outcomes = (
        long.groupby(KEY_COLUMNS, observed=True)
        .agg(
            TournamentGames=("TargetKey", "nunique"),
            TournamentWins=("Win", "sum"),
            TournamentAppearance=("TargetKey", lambda s: 1),
            TournamentLastDay=("DayNum", "max"),
        )
        .reset_index()
        .merge(final_teams, on=KEY_COLUMNS, how="left", validate="one_to_one")
        .merge(champions, on=KEY_COLUMNS, how="left", validate="one_to_one")
    )
    outcomes["ReachedFinal"] = outcomes["ReachedFinal"].fillna(0).astype("int8")
    outcomes["Champion"] = outcomes["Champion"].fillna(0).astype("int8")
    return outcomes


tournament_outcomes = tournament_team_outcomes(target_registry)
ensure_unique(tournament_outcomes, KEY_COLUMNS, "tournament team outcomes")


def build_program_priors(
    snapshot_keys: pd.DataFrame,
    outcomes: pd.DataFrame,
) -> pd.DataFrame:
    outcome_lookup = {
        (str(row.Gender), int(row.Season), int(row.TeamID)): row
        for row in outcomes.itertuples(index=False)
    }
    records: list[dict[str, Any]] = []

    for gender, gender_keys in snapshot_keys.groupby("Gender", sort=True, observed=True):
        history: defaultdict[int, dict[str, float | int | None]] = defaultdict(
            lambda: {
                "appearances": 0,
                "games": 0,
                "wins": 0,
                "finals": 0,
                "championships": 0,
                "last_appearance": None,
                "max_source_season": None,
            }
        )
        for season, season_keys in gender_keys.groupby("Season", sort=True, observed=True):
            season = int(season)
            for team_id in sorted(season_keys["TeamID"].astype(int).unique()):
                stats = history[int(team_id)]
                last = stats["last_appearance"]
                games = int(stats["games"])
                wins = int(stats["wins"])
                records.append(
                    {
                        "Gender": gender,
                        "Season": season,
                        "TeamID": int(team_id),
                        "prior__program_appearances": int(stats["appearances"]),
                        "prior__program_tournament_games": games,
                        "prior__program_tournament_wins": wins,
                        "prior__program_win_pct_smoothed": (
                            (wins + 1.0) / (games + 2.0)
                        ),
                        "prior__program_final_appearances": int(stats["finals"]),
                        "prior__program_championships": int(stats["championships"]),
                        "prior__program_years_since_appearance": (
                            season - int(last) if last is not None else np.nan
                        ),
                        "prior__program_has_tournament_history": int(games > 0),
                        "prior__program_max_source_season": (
                            int(stats["max_source_season"])
                            if stats["max_source_season"] is not None
                            else np.nan
                        ),
                    }
                )

            # Update only after every feature row for this season has been emitted.
            current = outcomes.loc[
                outcomes["Gender"].eq(gender)
                & outcomes["Season"].eq(season)
            ]
            for row in current.itertuples(index=False):
                stats = history[int(row.TeamID)]
                stats["appearances"] = int(stats["appearances"]) + 1
                stats["games"] = int(stats["games"]) + int(row.TournamentGames)
                stats["wins"] = int(stats["wins"]) + int(row.TournamentWins)
                stats["finals"] = int(stats["finals"]) + int(row.ReachedFinal)
                stats["championships"] = (
                    int(stats["championships"]) + int(row.Champion)
                )
                stats["last_appearance"] = season
                stats["max_source_season"] = season

    return pd.DataFrame(records)


program_features = build_program_priors(
    base_snapshot[KEY_COLUMNS].drop_duplicates(),
    tournament_outcomes,
)
ensure_unique(program_features, KEY_COLUMNS, "program priors")
assert (
    program_features.loc[
        program_features["prior__program_max_source_season"].notna(),
        "prior__program_max_source_season",
    ]
    < program_features.loc[
        program_features["prior__program_max_source_season"].notna(),
        "Season",
    ]
).all()

for column in program_features.columns:
    if column not in KEY_COLUMNS:
        register_feature(
            column,
            block="prior_program_history",
            universe="compact",
            source="historical NCAA tournament outcomes",
            temporal_rule="strictly earlier tournament seasons only",
            direction="higher_is_more_prior_experience",
        )

print("Program-prior feature table:", program_features.shape)
program_features.tail()


In [ ]:
coach_features = base_snapshot[KEY_COLUMNS].copy()

if FEATURE_CONFIG["computational_switches"]["run_coach_history"]:
    coaches = optional_frames["coaches"].copy()
    coaches["Gender"] = "M"
    active = coaches.loc[
        coaches["FirstDayNum"].le(CUTOFF_DAY)
        & coaches["LastDayNum"].ge(CUTOFF_DAY)
    ].copy()
    active = (
        active.sort_values(
            ["Season", "TeamID", "FirstDayNum", "LastDayNum", "CoachName"]
        )
        .drop_duplicates(["Season", "TeamID"], keep="last")
        .rename(columns={"CoachName": "CoachNameAtCutoff"})
    )
    active = active[
        ["Gender", "Season", "TeamID", "CoachNameAtCutoff", "FirstDayNum"]
    ]
    active["coach__midseason_change"] = active["FirstDayNum"].gt(0).astype("int8")
    active["coach__seasons_with_program_through_current"] = (
        active.sort_values(["TeamID", "CoachNameAtCutoff", "Season"])
        .groupby(["TeamID", "CoachNameAtCutoff"], observed=True)
        .cumcount()
        .add(1)
    )

    coach_season_outcomes = active.merge(
        tournament_outcomes,
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one",
    )
    for column in (
        "TournamentGames", "TournamentWins", "TournamentAppearance",
        "ReachedFinal", "Champion",
    ):
        coach_season_outcomes[column] = (
            coach_season_outcomes[column].fillna(0).astype(int)
        )

    coach_records: list[dict[str, Any]] = []
    history: defaultdict[str, dict[str, int | None]] = defaultdict(
        lambda: {
            "appearances": 0,
            "games": 0,
            "wins": 0,
            "finals": 0,
            "championships": 0,
            "last_appearance": None,
            "max_source_season": None,
        }
    )

    for season, season_rows in coach_season_outcomes.groupby(
        "Season", sort=True, observed=True
    ):
        season = int(season)
        for row in season_rows.itertuples(index=False):
            coach = str(row.CoachNameAtCutoff)
            stats = history[coach]
            games = int(stats["games"])
            wins = int(stats["wins"])
            last = stats["last_appearance"]
            coach_records.append(
                {
                    "Gender": "M",
                    "Season": season,
                    "TeamID": int(row.TeamID),
                    "coach__midseason_change": int(row.coach__midseason_change),
                    "coach__seasons_with_program_through_current": int(
                        row.coach__seasons_with_program_through_current
                    ),
                    "coach__prior_tournament_appearances": int(stats["appearances"]),
                    "coach__prior_tournament_games": games,
                    "coach__prior_tournament_wins": wins,
                    "coach__prior_tournament_win_pct_smoothed": (
                        (wins + 1.0) / (games + 2.0)
                    ),
                    "coach__prior_final_appearances": int(stats["finals"]),
                    "coach__prior_championships": int(stats["championships"]),
                    "coach__years_since_tournament_appearance": (
                        season - int(last) if last is not None else np.nan
                    ),
                    "coach__has_prior_tournament_history": int(games > 0),
                    "coach__max_source_season": (
                        int(stats["max_source_season"])
                        if stats["max_source_season"] is not None
                        else np.nan
                    ),
                }
            )

        # Aggregate by coach before updating in case of unusual duplicate assignments.
        updates = (
            season_rows.groupby("CoachNameAtCutoff", observed=True)
            .agg(
                TournamentAppearance=("TournamentAppearance", "max"),
                TournamentGames=("TournamentGames", "sum"),
                TournamentWins=("TournamentWins", "sum"),
                ReachedFinal=("ReachedFinal", "sum"),
                Champion=("Champion", "sum"),
            )
            .reset_index()
        )
        for row in updates.itertuples(index=False):
            coach = str(row.CoachNameAtCutoff)
            stats = history[coach]
            stats["appearances"] = (
                int(stats["appearances"]) + int(row.TournamentAppearance)
            )
            stats["games"] = int(stats["games"]) + int(row.TournamentGames)
            stats["wins"] = int(stats["wins"]) + int(row.TournamentWins)
            stats["finals"] = int(stats["finals"]) + int(row.ReachedFinal)
            stats["championships"] = (
                int(stats["championships"]) + int(row.Champion)
            )
            if int(row.TournamentAppearance) > 0:
                stats["last_appearance"] = season
                stats["max_source_season"] = season

    coach_features = pd.DataFrame(coach_records)
    ensure_unique(coach_features, KEY_COLUMNS, "coach priors")
    valid_source = coach_features["coach__max_source_season"].notna()
    assert (
        coach_features.loc[valid_source, "coach__max_source_season"]
        < coach_features.loc[valid_source, "Season"]
    ).all()

for column in coach_features.columns:
    if column not in KEY_COLUMNS:
        register_feature(
            column,
            block="prior_coach_history",
            universe="compact",
            availability="men_only",
            source="MTeamCoaches + historical NCAA outcomes",
            temporal_rule="coach active at DayNum 132; outcomes from strictly earlier seasons only",
            direction="higher_is_more_prior_experience",
        )

print("Coach-prior feature table:", coach_features.shape)
coach_features.tail()


## 9. Men's Massey ordinal consensus, disagreement, momentum, and stable systems

Massey data are not imputed into the women's feature universe. Stable systems are selected using
**availability through 2021 only**, never target performance. Lower ordinal ranks are converted to a
within-system strength percentile so systems with different field sizes remain comparable.


In [ ]:
massey_features = base_snapshot.loc[
    base_snapshot["Gender"].eq("M"), KEY_COLUMNS
].copy()
stable_massey_systems: list[str] = []

if FEATURE_CONFIG["computational_switches"]["run_massey_consensus"]:
    massey = optional_frames["massey"].copy()
    required = {"Season", "RankingDayNum", "SystemName", "TeamID", "OrdinalRank"}
    missing = required.difference(massey.columns)
    assert not missing, f"Massey table missing: {sorted(missing)}"

    massey = massey.loc[
        massey["RankingDayNum"].le(133)
        & massey["Season"].le(TARGET_SEASON)
    ].copy()
    massey["Gender"] = "M"
    massey["SystemName"] = massey["SystemName"].astype(str)

    latest = (
        massey.sort_values(
            ["Season", "SystemName", "TeamID", "RankingDayNum"]
        )
        .drop_duplicates(
            ["Season", "SystemName", "TeamID"],
            keep="last",
        )
    )
    latest["SystemTeamCount"] = (
        latest.groupby(
            ["Season", "SystemName"], observed=True
        )["TeamID"].transform("nunique")
    )
    latest["RankPercentile"] = safe_divide(
        latest["OrdinalRank"] - 1,
        latest["SystemTeamCount"] - 1,
    ).to_numpy()
    latest["MasseyStrength"] = 1.0 - latest["RankPercentile"]

    consensus = (
        latest.groupby(["Season", "TeamID"], observed=True)
        .agg(
            massey__systems=("SystemName", "nunique"),
            massey__latest_day_min=("RankingDayNum", "min"),
            massey__latest_day_max=("RankingDayNum", "max"),
            massey__ordinal_mean=("OrdinalRank", "mean"),
            massey__ordinal_median=("OrdinalRank", "median"),
            massey__ordinal_std=("OrdinalRank", "std"),
            massey__ordinal_min=("OrdinalRank", "min"),
            massey__ordinal_max=("OrdinalRank", "max"),
            massey__strength_mean=("MasseyStrength", "mean"),
            massey__strength_median=("MasseyStrength", "median"),
            massey__strength_std=("MasseyStrength", "std"),
            massey__strength_min=("MasseyStrength", "min"),
            massey__strength_max=("MasseyStrength", "max"),
            massey__strength_iqr=(
                "MasseyStrength",
                lambda s: float(s.quantile(0.75) - s.quantile(0.25)),
            ),
            massey__strength_trim05=(
                "MasseyStrength",
                lambda s: float(trim_mean(s.to_numpy(dtype=float), 0.05))
                if len(s) else np.nan,
            ),
        )
        .reset_index()
    )
    consensus["Gender"] = "M"

    # Early-to-late ranking movement, using the latest record available by DayNum 100.
    early = (
        massey.loc[massey["RankingDayNum"].le(100)]
        .sort_values(["Season", "SystemName", "TeamID", "RankingDayNum"])
        .drop_duplicates(["Season", "SystemName", "TeamID"], keep="last")
        [["Season", "SystemName", "TeamID", "OrdinalRank"]]
        .rename(columns={"OrdinalRank": "EarlyOrdinalRank"})
    )
    momentum = latest.merge(
        early,
        on=["Season", "SystemName", "TeamID"],
        how="left",
        validate="one_to_one",
    )
    momentum["RankImprovement"] = (
        momentum["EarlyOrdinalRank"] - momentum["OrdinalRank"]
    )
    momentum_summary = (
        momentum.groupby(["Season", "TeamID"], observed=True)
        .agg(
            massey__rank_improvement_mean=("RankImprovement", "mean"),
            massey__rank_improvement_median=("RankImprovement", "median"),
            massey__rank_improvement_std=("RankImprovement", "std"),
            massey__momentum_systems=("RankImprovement", "count"),
        )
        .reset_index()
    )
    momentum_summary["Gender"] = "M"

    # Stable-system selection is unsupervised and frozen using data through 2021 only.
    dev_latest = latest.loc[latest["Season"].le(DEVELOPMENT_LAST_SEASON)]
    system_availability = (
        dev_latest.groupby("SystemName", observed=True)
        .agg(
            DevelopmentSeasons=("Season", "nunique"),
            DevelopmentTeamSeasonRows=("TeamID", "size"),
            MedianTeamsPerSeason=("SystemTeamCount", "median"),
            LastDevelopmentSeason=("Season", "max"),
        )
        .reset_index()
        .sort_values(
            [
                "DevelopmentSeasons",
                "LastDevelopmentSeason",
                "MedianTeamsPerSeason",
                "DevelopmentTeamSeasonRows",
                "SystemName",
            ],
            ascending=[False, False, False, False, True],
        )
    )
    stable_massey_systems = system_availability.head(
        int(FEATURE_CONFIG["stable_massey_system_count"])
    )["SystemName"].tolist()
    system_availability["SelectedStableSystem"] = (
        system_availability["SystemName"].isin(stable_massey_systems)
    )
    system_availability.to_csv(
        FEATURE_REPORTS / "massey_system_availability.csv",
        index=False,
    )

    stable = latest.loc[
        latest["SystemName"].isin(stable_massey_systems)
    ].copy()
    safe_names: dict[str, str] = {}
    for system in stable_massey_systems:
        safe = re.sub(r"[^A-Za-z0-9]+", "_", str(system)).strip("_").lower()
        safe_names[system] = safe
    assert len(set(safe_names.values())) == len(safe_names), (
        "Stable Massey system names collide after sanitization."
    )
    stable["StableSystemColumn"] = stable["SystemName"].map(safe_names)
    stable_pivot = (
        stable.pivot_table(
            index=["Season", "TeamID"],
            columns="StableSystemColumn",
            values="MasseyStrength",
            aggfunc="last",
        )
        .add_prefix("massey__system_")
        .reset_index()
    )
    stable_pivot["Gender"] = "M"

    massey_features = (
        consensus.merge(
            momentum_summary,
            on=KEY_COLUMNS,
            how="outer",
            validate="one_to_one",
        )
        .merge(
            stable_pivot,
            on=KEY_COLUMNS,
            how="outer",
            validate="one_to_one",
        )
    )

ensure_unique(massey_features, KEY_COLUMNS, "Massey features")
for column in massey_features.columns:
    if column not in KEY_COLUMNS:
        register_feature(
            column,
            block="massey_consensus",
            universe="rich",
            availability="men_only",
            source="MMasseyOrdinals",
            temporal_rule="RankingDayNum <= 133; stable systems selected using availability through 2021",
            direction="higher_is_stronger_for_strength_and_improvement",
        )

print("Stable Massey systems:", stable_massey_systems)
print("Massey feature table:", massey_features.shape)
massey_features.tail()


## 10. Tournament seed metadata and strictly prequential hierarchical seed priors

Seed is known only for tournament-selected teams. The notebook preserves both a **seed-aware route**
and a **seed-free fallback route**.

Historical seed-matchup priors are target-derived, so they are generated prequentially: a matchup in
season `Y` can use only tournament outcomes from seasons `< Y`. Multiple pseudocount strengths are
stored for later nested selection.


In [ ]:
seed_features = seeds[
    ["Gender", "Season", "TeamID", "Seed", "SeedNum", "Region", "PlayIn"]
].drop_duplicates(KEY_COLUMNS)
ensure_unique(seed_features, KEY_COLUMNS, "seed metadata")
seed_features = seed_features.rename(
    columns={
        "Seed": "seed__raw",
        "SeedNum": "seed__number",
        "Region": "seed__region",
        "PlayIn": "seed__playin_code",
    }
)
seed_features["seed__available"] = 1
seed_features["seed__playin"] = (
    seed_features["seed__playin_code"].notna().astype("int8")
)
seed_features["seed__strength"] = 17.0 - seed_features["seed__number"].astype(float)
seed_features["seed__top4"] = seed_features["seed__number"].le(4).astype("int8")
seed_features["seed__top8"] = seed_features["seed__number"].le(8).astype("int8")

for column in (
    "seed__number", "seed__available", "seed__playin", "seed__strength",
    "seed__top4", "seed__top8",
):
    register_feature(
        column,
        block="selection_committee_prior",
        universe="seed_aware",
        source="NCAA tournament seeds",
        temporal_rule="Selection Sunday information for the prediction season",
        direction="higher_is_stronger_except_seed_number",
    )

print("Seed metadata:", seed_features.shape)
seed_features.tail()


In [ ]:
seed_lookup = seed_features[
    ["Gender", "Season", "TeamID", "seed__number"]
].copy()

seeded_targets = (
    target_registry.merge(
        seed_lookup.rename(
            columns={"TeamID": "Team1ID", "seed__number": "Team1Seed"}
        ),
        on=["Gender", "Season", "Team1ID"],
        how="left",
        validate="many_to_one",
    )
    .merge(
        seed_lookup.rename(
            columns={"TeamID": "Team2ID", "seed__number": "Team2Seed"}
        ),
        on=["Gender", "Season", "Team2ID"],
        how="left",
        validate="many_to_one",
    )
)
seeded_targets["BothSeeds"] = (
    seeded_targets["Team1Seed"].notna()
    & seeded_targets["Team2Seed"].notna()
)
seeded_targets["FavoriteSeed"] = seeded_targets[
    ["Team1Seed", "Team2Seed"]
].min(axis=1)
seeded_targets["UnderdogSeed"] = seeded_targets[
    ["Team1Seed", "Team2Seed"]
].max(axis=1)
seeded_targets["SeedGapAbs"] = (
    seeded_targets["Team1Seed"] - seeded_targets["Team2Seed"]
).abs()
seeded_targets["Team1IsFavorite"] = (
    seeded_targets["Team1Seed"] < seeded_targets["Team2Seed"]
)
seeded_targets["EqualSeeds"] = (
    seeded_targets["Team1Seed"] == seeded_targets["Team2Seed"]
)
seeded_targets["FavoriteWon"] = np.where(
    seeded_targets["EqualSeeds"] | ~seeded_targets["BothSeeds"],
    np.nan,
    np.where(
        seeded_targets["Team1IsFavorite"],
        seeded_targets["Team1Win"],
        1 - seeded_targets["Team1Win"],
    ),
)


def prior_counts(
    history: pd.DataFrame,
    keys: list[str],
) -> dict[tuple[Any, ...], tuple[float, int]]:
    if history.empty:
        return {}
    grouped = (
        history.dropna(subset=["FavoriteWon"])
        .groupby(keys, observed=True)["FavoriteWon"]
        .agg(["sum", "count"])
    )
    return {
        tuple(index) if isinstance(index, tuple) else (index,): (
            float(row["sum"]),
            int(row["count"]),
        )
        for index, row in grouped.iterrows()
    }


def get_count(
    mapping: dict[tuple[Any, ...], tuple[float, int]],
    key: tuple[Any, ...],
) -> tuple[float, int]:
    return mapping.get(tuple(key), (0.0, 0))


def build_prequential_seed_priors(
    requests: pd.DataFrame,
    history_targets: pd.DataFrame,
) -> pd.DataFrame:
    req = requests.copy()
    req["RequestKeyInternal"] = np.arange(len(req), dtype=np.int64)
    req["HasOutcomeInternal"] = req.get(
        "Team1Win", pd.Series(np.nan, index=req.index)
    ).notna()

    records: list[dict[str, Any]] = []
    for season in sorted(req["Season"].astype(int).unique()):
        history = history_targets.loc[
            history_targets["Season"].lt(int(season))
            & history_targets["BothSeeds"]
            & ~history_targets["EqualSeeds"]
        ].copy()

        gender_pair = prior_counts(
            history,
            ["Gender", "FavoriteSeed", "UnderdogSeed"],
        )
        gender_gap = prior_counts(
            history,
            ["Gender", "SeedGapAbs"],
        )
        gender_favorite_seed = prior_counts(
            history,
            ["Gender", "FavoriteSeed"],
        )
        gender_all = prior_counts(history, ["Gender"])
        pooled_pair = prior_counts(
            history,
            ["FavoriteSeed", "UnderdogSeed"],
        )
        pooled_gap = prior_counts(history, ["SeedGapAbs"])
        pooled_all = prior_counts(
            history.assign(PooledKey="all"), ["PooledKey"]
        )

        season_requests = req.loc[req["Season"].eq(int(season))]
        for row in season_requests.itertuples(index=False):
            both = pd.notna(row.Team1Seed) and pd.notna(row.Team2Seed)
            equal = bool(both and row.Team1Seed == row.Team2Seed)
            record: dict[str, Any] = {
                "RequestKeyInternal": int(row.RequestKeyInternal),
                "seedprior__max_source_season": (
                    int(history["Season"].max()) if not history.empty else np.nan
                ),
                "seedprior__history_games": int(len(history)),
            }

            if not both or equal:
                for pseudocount in FEATURE_CONFIG["seed_prior_pseudocounts"]:
                    tag = str(pseudocount).replace(".", "p")
                    record[f"seedprior__team1_prob_pc{tag}"] = 0.5
                    record[f"seedprior__favorite_prob_pc{tag}"] = 0.5
                    record[f"seedprior__effective_games_pc{tag}"] = 0
                record["seedprior__exact_pair_games"] = 0
                record["seedprior__gap_games"] = 0
                record["seedprior__favorite_seed_games"] = 0
                record["seedprior__generic_gender_games"] = 0
                records.append(record)
                continue

            favorite_seed = float(min(row.Team1Seed, row.Team2Seed))
            underdog_seed = float(max(row.Team1Seed, row.Team2Seed))
            gap = float(abs(row.Team1Seed - row.Team2Seed))
            team1_favorite = bool(row.Team1Seed < row.Team2Seed)

            pair_success, pair_n = get_count(
                gender_pair, (row.Gender, favorite_seed, underdog_seed)
            )
            gap_success, gap_n = get_count(
                gender_gap, (row.Gender, gap)
            )
            favorite_success, favorite_n = get_count(
                gender_favorite_seed, (row.Gender, favorite_seed)
            )
            generic_success, generic_n = get_count(
                gender_all, (row.Gender,)
            )
            pooled_pair_success, pooled_pair_n = get_count(
                pooled_pair, (favorite_seed, underdog_seed)
            )
            pooled_gap_success, pooled_gap_n = get_count(
                pooled_gap, (gap,)
            )
            pooled_success, pooled_n = get_count(
                pooled_all, ("all",)
            )

            record["seedprior__exact_pair_games"] = pair_n
            record["seedprior__gap_games"] = gap_n
            record["seedprior__favorite_seed_games"] = favorite_n
            record["seedprior__generic_gender_games"] = generic_n
            record["seedprior__pooled_pair_games"] = pooled_pair_n
            record["seedprior__pooled_gap_games"] = pooled_gap_n

            for pseudocount in FEATURE_CONFIG["seed_prior_pseudocounts"]:
                tag = str(pseudocount).replace(".", "p")
                # A hierarchy of increasingly broad estimates; evidence controls weight.
                generic = beta_smoothed_rate(
                    generic_success, generic_n, pseudocount
                ) if generic_n else (
                    beta_smoothed_rate(pooled_success, pooled_n, pseudocount)
                    if pooled_n else 0.5
                )
                pooled_gap_rate = (
                    beta_smoothed_rate(
                        pooled_gap_success, pooled_gap_n, pseudocount
                    )
                    if pooled_gap_n else generic
                )
                gap_rate = (
                    beta_smoothed_rate(gap_success, gap_n, pseudocount)
                    if gap_n else pooled_gap_rate
                )
                favorite_rate = (
                    beta_smoothed_rate(
                        favorite_success, favorite_n, pseudocount
                    )
                    if favorite_n else generic
                )
                pooled_pair_rate = (
                    beta_smoothed_rate(
                        pooled_pair_success, pooled_pair_n, pseudocount
                    )
                    if pooled_pair_n else gap_rate
                )
                pair_rate = (
                    beta_smoothed_rate(pair_success, pair_n, pseudocount)
                    if pair_n else pooled_pair_rate
                )

                # Empirical reliability weighting without looking at current outcomes.
                pair_weight = pair_n / (pair_n + 20.0)
                gap_weight = gap_n / (gap_n + 40.0)
                favorite_weight = favorite_n / (favorite_n + 40.0)
                broad = (
                    0.50 * gap_rate
                    + 0.30 * favorite_rate
                    + 0.20 * generic
                )
                favorite_probability = (
                    pair_weight * pair_rate
                    + (1.0 - pair_weight)
                    * (
                        gap_weight * gap_rate
                        + (1.0 - gap_weight)
                        * (
                            favorite_weight * favorite_rate
                            + (1.0 - favorite_weight) * broad
                        )
                    )
                )
                favorite_probability = float(
                    np.clip(favorite_probability, 0.01, 0.99)
                )
                record[f"seedprior__favorite_prob_pc{tag}"] = favorite_probability
                record[f"seedprior__team1_prob_pc{tag}"] = (
                    favorite_probability
                    if team1_favorite
                    else 1.0 - favorite_probability
                )
                record[f"seedprior__effective_games_pc{tag}"] = (
                    pair_n + gap_n + favorite_n
                )
            records.append(record)

    result = req.merge(
        pd.DataFrame(records),
        on="RequestKeyInternal",
        how="left",
        validate="one_to_one",
    )
    return result.sort_values("RequestKeyInternal").reset_index(drop=True)


historical_seed_prior_source = seeded_targets[
    [
        "Gender", "Season", "Team1ID", "Team2ID", "Team1Win",
        "Team1Seed", "Team2Seed", "FavoriteSeed", "UnderdogSeed",
        "SeedGapAbs", "Team1IsFavorite", "EqualSeeds", "BothSeeds",
        "FavoriteWon",
    ]
].copy()

historical_seed_priors = build_prequential_seed_priors(
    historical_seed_prior_source,
    seeded_targets,
)

for column in historical_seed_priors.columns:
    if column.startswith("seedprior__"):
        register_feature(
            column,
            block="prequential_seed_priors",
            universe="seed_aware",
            source="historical NCAA outcomes + current-season seeds",
            temporal_rule="outcomes from seasons strictly earlier than each request season",
            direction="probability_or_evidence_count",
        )

valid_prior_source = historical_seed_priors["seedprior__max_source_season"].notna()
assert (
    historical_seed_priors.loc[
        valid_prior_source, "seedprior__max_source_season"
    ]
    < historical_seed_priors.loc[valid_prior_source, "Season"]
).all()

print("Historical prequential seed-prior rows:", historical_seed_priors.shape)
historical_seed_priors.tail()


## 11. Geography availability and schedule-dispersion features

The competition supplies game cities but not team home coordinates. This notebook therefore creates
only reproducible city-dispersion features from supplied IDs. Distance, altitude, and travel burden are
registered as external extensions rather than fabricated.


In [ ]:
geography_features = base_snapshot[KEY_COLUMNS].copy()

if "game_cities" in optional_frames:
    game_cities = optional_frames["game_cities"].copy()
    regular_cities = game_cities.loc[
        game_cities["CRType"].eq("Regular")
        & game_cities["DayNum"].le(CUTOFF_DAY)
    ].copy()
    regular_cities["Team1ID"] = regular_cities[
        ["WTeamID", "LTeamID"]
    ].min(axis=1)
    regular_cities["Team2ID"] = regular_cities[
        ["WTeamID", "LTeamID"]
    ].max(axis=1)
    regular_cities["GameKey"] = (
        regular_cities["Gender"].astype(str)
        + "_"
        + regular_cities["Season"].astype(str)
        + "_"
        + regular_cities["DayNum"].astype(str)
        + "_"
        + regular_cities["Team1ID"].astype(str)
        + "_"
        + regular_cities["Team2ID"].astype(str)
    )
    city_by_game = regular_cities[
        ["Gender", "Season", "GameKey", "CityID"]
    ].drop_duplicates("GameKey")
    city_team = team_games[
        ["Gender", "Season", "GameKey", "TeamID", "TeamLoc"]
    ].merge(
        city_by_game,
        on=["Gender", "Season", "GameKey"],
        how="left",
        validate="many_to_one",
    )
    city_team["CityAvailable"] = city_team["CityID"].notna()
    city_team["NeutralCityAvailable"] = (
        city_team["TeamLoc"].eq("N") & city_team["CityAvailable"]
    )
    geography_features = (
        city_team.groupby(KEY_COLUMNS, observed=True)
        .agg(
            geography__games_with_city=("CityAvailable", "sum"),
            geography__city_coverage=("CityAvailable", "mean"),
            geography__distinct_cities=("CityID", "nunique"),
            geography__neutral_games_with_city=("NeutralCityAvailable", "sum"),
        )
        .reset_index()
    )
    geography_features["geography__games_per_distinct_city"] = safe_divide(
        geography_features["geography__games_with_city"],
        geography_features["geography__distinct_cities"],
    ).to_numpy()

ensure_unique(geography_features, KEY_COLUMNS, "geography features")
for column in geography_features.columns:
    if column not in KEY_COLUMNS:
        register_feature(
            column,
            block="geography_and_schedule_dispersion",
            universe="compact",
            source="MGameCities/WGameCities",
            temporal_rule="regular-season game cities through DayNum 132",
            direction="context_dependent",
        )

external_contract = pd.DataFrame(
    [
        {
            "FeatureFamily": "travel_distance_and_time_zones",
            "Status": "not_built",
            "RequiredData": "timestamped team home coordinates and venue latitude/longitude",
            "ScientificRequirement": "historically correct location as of each season; no current-only geocoding assumptions",
        },
        {
            "FeatureFamily": "altitude_and_environment",
            "Status": "not_built",
            "RequiredData": "venue altitude and historically correct game location",
            "ScientificRequirement": "source version and timestamp must be preserved",
        },
        {
            "FeatureFamily": "player_availability_and_injuries",
            "Status": "not_built",
            "RequiredData": "historical timestamped rosters, minutes, injuries, suspensions",
            "ScientificRequirement": "only information known before each historical tournament",
        },
        {
            "FeatureFamily": "betting_and_prediction_markets",
            "Status": "not_built",
            "RequiredData": "timestamped pregame or pre-tournament market probabilities",
            "ScientificRequirement": "historical snapshots; no closing-line leakage for earlier forecast dates",
        },
        {
            "FeatureFamily": "external_power_ratings",
            "Status": "not_built",
            "RequiredData": "historical KenPom/BartTorvik/NET/AP snapshots",
            "ScientificRequirement": "licensed/reproducible source with as-of dates for every validation season",
        },
    ]
)
external_contract.to_csv(
    FEATURE_REPORTS / "external_feature_extension_contract.csv",
    index=False,
)

print("Geography feature table:", geography_features.shape)
external_contract


## 12. Assemble the team-season candidate feature store

The store keeps source metadata and every candidate block. It then adds within-season standardized
scores and percentiles for a curated set of strength measures, allowing models to compare teams across
eras without relying only on raw scales.


In [ ]:
base_metadata_columns = [
    column for column in [
        "Gender", "Season", "TeamID", "TeamName", "ConfAbbrev",
        "SnapshotDayNum", "CompactGames", "DetailedGames",
        "DetailedCoverageRate", "HasAnyDetailedData",
        "HasCompleteDetailedCoverage", "RichFeatureCoverageEligible",
        "FlaggedPossessionRate", "FirstD1Season", "LastD1Season",
    ]
    if column in base_snapshot.columns
]
team_feature_store = base_snapshot[base_metadata_columns].copy()
ensure_unique(team_feature_store, KEY_COLUMNS, "base metadata")

for name, block in (
    ("compact", compact_features),
    ("detailed", detailed_features),
    ("ratings", rating_features),
    ("adjusted", adjusted_features),
    ("schedule", schedule_features),
    ("conference", conference_team_features),
    ("program", program_features),
    ("coach", coach_features),
    ("massey", massey_features),
    ("geography", geography_features),
):
    print("Merging feature block:", name, block.shape)
    team_feature_store = merge_feature_block(
        team_feature_store, block, name=name
    )

team_feature_store = team_feature_store.merge(
    seed_features,
    on=KEY_COLUMNS,
    how="left",
    validate="one_to_one",
)
team_feature_store["seed__available"] = (
    team_feature_store["seed__available"].fillna(0).astype("int8")
)
team_feature_store["seed__playin"] = (
    team_feature_store["seed__playin"].fillna(0).astype("int8")
)
team_feature_store["availability__detailed"] = (
    team_feature_store.get(
        "HasAnyDetailedData",
        pd.Series(False, index=team_feature_store.index),
    )
    .fillna(False)
    .astype("int8")
)
team_feature_store["availability__rich_coverage"] = (
    team_feature_store.get(
        "RichFeatureCoverageEligible",
        pd.Series(False, index=team_feature_store.index),
    )
    .fillna(False)
    .astype("int8")
)
team_feature_store["availability__massey"] = (
    team_feature_store["Gender"].eq("M")
    & team_feature_store.get(
        "massey__systems",
        pd.Series(np.nan, index=team_feature_store.index),
    ).notna()
).astype("int8")
team_feature_store["availability__coach_history"] = (
    team_feature_store["Gender"].eq("M")
    & team_feature_store.get(
        "coach__has_prior_tournament_history",
        pd.Series(np.nan, index=team_feature_store.index),
    ).notna()
).astype("int8")
team_feature_store["context__season_centered_2000"] = (
    team_feature_store["Season"] - 2000
)
team_feature_store["context__d1_tenure"] = np.where(
    team_feature_store.get(
        "FirstD1Season",
        pd.Series(np.nan, index=team_feature_store.index),
    ).notna(),
    team_feature_store["Season"] - team_feature_store["FirstD1Season"],
    np.nan,
)
team_feature_store["context__women"] = team_feature_store["Gender"].eq("W").astype("int8")

# Add within-gender-season z-scores and percentiles for selected core strength measures.
normalization_candidates = [
    "compact__win_pct",
    "compact__margin_mean",
    "compact__margin_trim02",
    "detailed__all__net_rtg_ros",
    "detailed__clean__net_rtg_ros",
    "detailed__all__efg_ros",
    "schedule__rpi",
    "schedule__opponent_strength_mean",
    "schedule__quality_win_points_sum",
    "conference__strength_mean",
    "adj__net_rtg_a10p0",
    "adj__net_rtg_a50p0",
    "rating__elo_mov_538",
    "rating__margin_ridge_a25p0",
    "rating__bt_c1p0",
    "rating__colley",
    "rating__pagerank_margin",
    "massey__strength_mean",
]
for column in normalization_candidates:
    if column not in team_feature_store.columns:
        continue
    grouped = team_feature_store.groupby(
        ["Gender", "Season"], observed=True
    )[column]
    mean = grouped.transform("mean")
    std = grouped.transform("std").replace(0, np.nan)
    team_feature_store[f"norm__{column}__z"] = (
        team_feature_store[column] - mean
    ) / std
    team_feature_store[f"norm__{column}__pct"] = grouped.rank(
        pct=True, method="average"
    )
    register_feature(
        f"norm__{column}__z",
        block="within_season_normalization",
        universe=FEATURE_META.get(column, {}).get("Universe", "compact"),
        availability=FEATURE_META.get(column, {}).get("Availability", "common"),
        source=column,
        direction="higher_follows_source",
    )
    register_feature(
        f"norm__{column}__pct",
        block="within_season_normalization",
        universe=FEATURE_META.get(column, {}).get("Universe", "compact"),
        availability=FEATURE_META.get(column, {}).get("Availability", "common"),
        source=column,
        direction="higher_follows_source",
    )

for column in (
    "availability__detailed", "availability__rich_coverage",
    "availability__massey", "availability__coach_history",
    "context__season_centered_2000", "context__d1_tenure", "context__women",
):
    register_feature(
        column,
        block="availability_and_context",
        universe="compact",
        availability="common" if "massey" not in column and "coach" not in column else "men_only",
        source="snapshot/source metadata",
        temporal_rule="known at forecast cutoff",
        direction="context_dependent",
    )

team_feature_store["TeamFeatureKey"] = (
    team_feature_store["Gender"].astype(str)
    + "_"
    + team_feature_store["Season"].astype(str)
    + "_"
    + team_feature_store["TeamID"].astype(str)
)
ensure_unique(team_feature_store, KEY_COLUMNS, "team feature store")
assert team_feature_store["TeamFeatureKey"].is_unique

print("Team feature store:", team_feature_store.shape)
print("Candidate feature metadata records:", len(FEATURE_META))
team_feature_store.tail()


## 13. Build historical and Stage 2 matchup feature stores

Every matchup is represented in the competition's lower-TeamID direction. Team-level candidate
features are converted to Team1-minus-Team2 differences. Absolute gaps and selected matchup-level means
are retained only where they add a distinct hypothesis. The notebook also creates basketball-motivated
interaction edges rather than an indiscriminate polynomial explosion.


In [ ]:
# Decide which team-level fields are legitimate model candidates.
def team_feature_is_candidate(column: str) -> bool:
    if column not in FEATURE_META:
        return False
    blocked_tokens = (
        "max_source_season",
        "_home_effect",
        "_league_off_rtg",
        "latest_day_",
    )
    if any(token in column for token in blocked_tokens):
        return False
    if column == "context__women":
        return False  # matchup-level gender term is added directly.
    return pd.api.types.is_numeric_dtype(team_feature_store[column]) or pd.api.types.is_bool_dtype(
        team_feature_store[column]
    )


TEAM_MODEL_FEATURES = [
    column
    for column in team_feature_store.columns
    if team_feature_is_candidate(column)
]

ABSDIFF_TEAM_FEATURES = [
    column for column in TEAM_MODEL_FEATURES
    if any(
        token in column
        for token in (
            "margin", "win_pct", "pythag", "net_rtg", "off_rtg", "def_rtg",
            "efg", "tov_pct", "orb_pct", "drb_pct", "ft_rate", "pace",
            "rating__", "adj__", "schedule__", "massey__", "seed__number",
            "coverage", "std", "mad", "iqr", "days_since", "rest_days",
        )
    )
]

MEAN_TEAM_FEATURES = [
    column for column in TEAM_MODEL_FEATURES
    if any(
        token in column
        for token in (
            "games", "coverage", "flagged", "pace", "std", "mad", "iqr",
            "program_", "coach_", "massey__systems", "days_since", "rest_days",
        )
    )
]


def join_two_teams(
    requests: pd.DataFrame,
    team_store: pd.DataFrame,
) -> pd.DataFrame:
    metadata = requests.copy()
    metadata["RequestOrderInternal"] = np.arange(len(metadata), dtype=np.int64)
    source = team_store[KEY_COLUMNS + TEAM_MODEL_FEATURES].copy()

    team1 = source.rename(
        columns={
            "TeamID": "Team1ID",
            **{column: f"T1__{column}" for column in TEAM_MODEL_FEATURES},
        }
    )
    team2 = source.rename(
        columns={
            "TeamID": "Team2ID",
            **{column: f"T2__{column}" for column in TEAM_MODEL_FEATURES},
        }
    )
    joined = (
        metadata.merge(
            team1,
            on=["Gender", "Season", "Team1ID"],
            how="left",
            validate="many_to_one",
        )
        .merge(
            team2,
            on=["Gender", "Season", "Team2ID"],
            how="left",
            validate="many_to_one",
        )
        .sort_values("RequestOrderInternal")
        .reset_index(drop=True)
    )
    return joined


def available_pair_value(
    joined: pd.DataFrame,
    feature: str,
    side: str,
) -> pd.Series:
    column = f"{side}__{feature}"
    if column not in joined.columns:
        return pd.Series(np.nan, index=joined.index, dtype=float)
    return pd.to_numeric(
        joined[column], errors="coerce"
    ).astype("float64")


def add_basketball_interactions(
    output: pd.DataFrame,
    joined: pd.DataFrame,
) -> pd.DataFrame:
    result = output.copy()

    def interaction(
        name: str,
        t1_offense: str,
        t1_defense_allowed: str,
    ) -> None:
        t1_attack = available_pair_value(joined, t1_offense, "T1")
        t2_attack = available_pair_value(joined, t1_offense, "T2")
        t1_allowed = available_pair_value(joined, t1_defense_allowed, "T1")
        t2_allowed = available_pair_value(joined, t1_defense_allowed, "T2")
        # Positive means the Team1 attack/defense combination is favored.
        result[name] = (
            (t1_attack - t2_allowed) - (t2_attack - t1_allowed)
        ).astype("float32")
        register_feature(
            name,
            block="matchup_interactions",
            universe="rich",
            source=f"{t1_offense} versus {t1_defense_allowed}",
            temporal_rule="same legal team snapshots for both sides",
            direction="positive_favors_team1",
        )

    interaction(
        "interaction__adjusted_offense_defense_edge",
        "adj__off_rtg_a50p0",
        "adj__def_rtg_a50p0",
    )
    interaction(
        "interaction__shooting_attack_defense_edge",
        "detailed__all__efg_ros",
        "detailed__all__opp_efg_ros",
    )
    interaction(
        "interaction__clean_shooting_attack_defense_edge",
        "detailed__clean__efg_ros",
        "detailed__clean__opp_efg_ros",
    )

    t1_tov = available_pair_value(joined, "detailed__all__tov_pct_ros", "T1")
    t2_tov = available_pair_value(joined, "detailed__all__tov_pct_ros", "T2")
    t1_forced = available_pair_value(joined, "detailed__all__opp_tov_pct_ros", "T1")
    t2_forced = available_pair_value(joined, "detailed__all__opp_tov_pct_ros", "T2")
    result["interaction__turnover_pressure_edge"] = (
        (t2_tov - t1_forced) - (t1_tov - t2_forced)
    ).astype("float32")

    t1_orb = available_pair_value(joined, "detailed__all__orb_pct_ros", "T1")
    t2_orb = available_pair_value(joined, "detailed__all__orb_pct_ros", "T2")
    t1_drb = available_pair_value(joined, "detailed__all__drb_pct_ros", "T1")
    t2_drb = available_pair_value(joined, "detailed__all__drb_pct_ros", "T2")
    result["interaction__rebounding_matchup_edge"] = (
        (t1_orb + t1_drb) - (t2_orb + t2_drb)
    ).astype("float32")

    t1_ft = available_pair_value(joined, "detailed__all__ft_rate_ros", "T1")
    t2_ft = available_pair_value(joined, "detailed__all__ft_rate_ros", "T2")
    t1_opp_ft = available_pair_value(joined, "detailed__all__opp_ft_rate_ros", "T1")
    t2_opp_ft = available_pair_value(joined, "detailed__all__opp_ft_rate_ros", "T2")
    # The exact fallback column spelling is handled below.
    if t1_opp_ft.isna().all():
        t1_opp_ft = available_pair_value(joined, "detailed__all__oppftrate_mean", "T1")
        t2_opp_ft = available_pair_value(joined, "detailed__all__oppftrate_mean", "T2")
    result["interaction__free_throw_pressure_edge"] = (
        (t1_ft - t2_opp_ft) - (t2_ft - t1_opp_ft)
    ).astype("float32")

    t1_pace = available_pair_value(joined, "detailed__all__pace_ros", "T1")
    t2_pace = available_pair_value(joined, "detailed__all__pace_ros", "T2")
    result["interaction__pace_gap"] = (t1_pace - t2_pace).abs().astype("float32")
    result["interaction__pace_mean"] = ((t1_pace + t2_pace) / 2.0).astype("float32")

    t1_vol = available_pair_value(joined, "compact__margin_std", "T1")
    t2_vol = available_pair_value(joined, "compact__margin_std", "T2")
    result["interaction__combined_margin_volatility"] = (
        np.sqrt(np.square(t1_vol) + np.square(t2_vol))
    ).astype("float32")

    t1_strength = available_pair_value(joined, SCHEDULE_RATING, "T1")
    t2_strength = available_pair_value(joined, SCHEDULE_RATING, "T2")
    result["interaction__strength_gap_squared_signed"] = (
        np.sign(t1_strength - t2_strength)
        * np.square(t1_strength - t2_strength)
    ).astype("float32")

    for column in (
        "interaction__turnover_pressure_edge",
        "interaction__rebounding_matchup_edge",
        "interaction__free_throw_pressure_edge",
        "interaction__pace_gap",
        "interaction__pace_mean",
        "interaction__combined_margin_volatility",
        "interaction__strength_gap_squared_signed",
    ):
        register_feature(
            column,
            block="matchup_interactions",
            universe="rich",
            source="paired team-season feature store",
            temporal_rule="same legal team snapshots for both sides",
            direction="positive_favors_team1_or_context",
        )
    return result


def build_matchup_store(
    requests: pd.DataFrame,
    team_store: pd.DataFrame,
) -> pd.DataFrame:
    joined = join_two_teams(requests, team_store)
    metadata_columns = [
        column for column in requests.columns
        if column not in TEAM_MODEL_FEATURES
    ]
    metadata_frame = joined[
        [column for column in metadata_columns if column in joined.columns]
    ].reset_index(drop=True)

    pair_data: dict[str, pd.Series] = {}
    for feature in TEAM_MODEL_FEATURES:
        t1 = available_pair_value(joined, feature, "T1")
        t2 = available_pair_value(joined, feature, "T2")
        pair_data[f"diff__{feature}"] = (t1 - t2).astype("float32")
        if feature in ABSDIFF_TEAM_FEATURES:
            pair_data[f"absdiff__{feature}"] = (t1 - t2).abs().astype("float32")
        if feature in MEAN_TEAM_FEATURES:
            pair_data[f"mean__{feature}"] = ((t1 + t2) / 2.0).astype("float32")

    pair_data["context__women"] = joined["Gender"].eq("W").astype("int8")
    pair_data["context__season_centered_2000"] = (
        joined["Season"].astype(int) - 2000
    ).astype("int16")

    # Explicit seed context from team-level metadata.
    t1_seed = available_pair_value(joined, "seed__number", "T1")
    t2_seed = available_pair_value(joined, "seed__number", "T2")
    pair_data["matchup__both_seeds_available"] = (
        t1_seed.notna() & t2_seed.notna()
    ).astype("int8")
    pair_data["matchup__team1_seed"] = t1_seed.astype("float32")
    pair_data["matchup__team2_seed"] = t2_seed.astype("float32")
    pair_data["matchup__seed_diff"] = (t1_seed - t2_seed).astype("float32")
    pair_data["matchup__seed_gap_abs"] = (t1_seed - t2_seed).abs().astype("float32")
    pair_data["matchup__team1_is_seed_favorite"] = (
        t1_seed.lt(t2_seed)
    ).astype("int8")
    pair_data["matchup__equal_seed"] = (
        t1_seed.eq(t2_seed) & t1_seed.notna()
    ).astype("int8")

    output = pd.concat(
        [metadata_frame, pd.DataFrame(pair_data).reset_index(drop=True)],
        axis=1,
    )
    output = add_basketball_interactions(output, joined)

    assert len(output) == len(requests)
    return output


historical_matchups = build_matchup_store(
    target_registry,
    team_feature_store,
)
stage2_matchups = build_matchup_store(
    submission_routing,
    team_feature_store,
)

print("Historical matchup store before priors:", historical_matchups.shape)
print("Stage 2 matchup store before priors:", stage2_matchups.shape)


In [ ]:
# Attach prequential priors to historical matchups.
seed_prior_columns = [
    column for column in historical_seed_priors.columns
    if column.startswith("seedprior__")
]
historical_prior_block = historical_seed_priors[
    ["Gender", "Season", "Team1ID", "Team2ID"] + seed_prior_columns
].copy()
ensure_unique(
    historical_prior_block,
    ["Gender", "Season", "Team1ID", "Team2ID"],
    "historical seed priors",
)
historical_matchups = historical_matchups.merge(
    historical_prior_block,
    on=["Gender", "Season", "Team1ID", "Team2ID"],
    how="left",
    validate="one_to_one",
)

# Build 2026 priors from all tournament outcomes through 2025.
stage2_seed_source = (
    submission_routing.merge(
        seed_lookup.rename(
            columns={"TeamID": "Team1ID", "seed__number": "Team1Seed"}
        ),
        on=["Gender", "Season", "Team1ID"],
        how="left",
        validate="many_to_one",
    )
    .merge(
        seed_lookup.rename(
            columns={"TeamID": "Team2ID", "seed__number": "Team2Seed"}
        ),
        on=["Gender", "Season", "Team2ID"],
        how="left",
        validate="many_to_one",
    )
)
stage2_seed_priors = build_prequential_seed_priors(
    stage2_seed_source,
    seeded_targets,
)
stage2_prior_block = stage2_seed_priors[
    ["Gender", "Season", "Team1ID", "Team2ID"] + seed_prior_columns
].copy()
ensure_unique(
    stage2_prior_block,
    ["Gender", "Season", "Team1ID", "Team2ID"],
    "Stage 2 seed priors",
)
stage2_matchups = stage2_matchups.merge(
    stage2_prior_block,
    on=["Gender", "Season", "Team1ID", "Team2ID"],
    how="left",
    validate="one_to_one",
)

historical_matchups["FeatureRoute"] = np.where(
    historical_matchups["matchup__both_seeds_available"].eq(1),
    "seed_aware",
    "seed_free_fallback",
)

# Register pairwise transformations.
for feature in TEAM_MODEL_FEATURES:
    source_meta = FEATURE_META[feature]
    register_feature(
        f"diff__{feature}",
        block=source_meta["Block"],
        universe=source_meta["Universe"],
        availability=source_meta["Availability"],
        source=feature,
        temporal_rule=source_meta["TemporalRule"],
        direction="positive_means_team1_higher",
        notes="Team1 minus Team2.",
    )
    if feature in ABSDIFF_TEAM_FEATURES:
        register_feature(
            f"absdiff__{feature}",
            block=source_meta["Block"],
            universe=source_meta["Universe"],
            availability=source_meta["Availability"],
            source=feature,
            temporal_rule=source_meta["TemporalRule"],
            direction="larger_means_greater_matchup_separation",
            notes="Absolute Team1-Team2 gap.",
        )
    if feature in MEAN_TEAM_FEATURES:
        register_feature(
            f"mean__{feature}",
            block=source_meta["Block"],
            universe=source_meta["Universe"],
            availability=source_meta["Availability"],
            source=feature,
            temporal_rule=source_meta["TemporalRule"],
            direction="matchup_context",
            notes="Mean of both teams.",
        )

for column in (
    "context__women",
    "context__season_centered_2000",
    "matchup__both_seeds_available",
    "matchup__team1_seed",
    "matchup__team2_seed",
    "matchup__seed_diff",
    "matchup__seed_gap_abs",
    "matchup__team1_is_seed_favorite",
    "matchup__equal_seed",
):
    register_feature(
        column,
        block="matchup_context",
        universe="seed_aware" if "seed" in column else "compact",
        source="paired team snapshots / seed metadata",
        temporal_rule="known before tournament",
        direction="context_or_positive_favors_team1",
    )

assert len(historical_matchups) == len(target_registry)
assert len(stage2_matchups) == len(submission_routing)
assert historical_matchups["TargetKey"].is_unique
assert stage2_matchups["ID"].is_unique
assert (historical_matchups["Team1ID"] < historical_matchups["Team2ID"]).all()
assert (stage2_matchups["Team1ID"] < stage2_matchups["Team2ID"]).all()

print("Historical matchup feature store:", historical_matchups.shape)
print("Stage 2 matchup feature store:", stage2_matchups.shape)
historical_matchups.groupby(
    ["DatasetRole", "Gender", "FeatureRoute"], observed=True
).size().to_frame("Rows")


## 14. Define architecture- and route-specific candidate sets

Feature selection has **not** occurred. These lists only enforce source availability:

- women's and pooled models cannot use men's-only Massey or coach fields;
- seed-free models cannot use seed or seed-prior fields;
- compact models cannot use detailed/adjusted-efficiency fields;
- rich models may use compact and rich features;
- identifiers and targets are never candidates.


In [ ]:
TARGET_AND_ID_COLUMNS = {
    "TargetKey", "GameKey", "ID", "Team1Win", "Team1Margin",
    "Team1Score", "Team2Score", "WTeamID", "LTeamID", "WScore", "LScore",
    "DayNum", "Team1Loc", "WLoc", "NumOT", "DatasetRole",
    "CompactUniverseEligible", "RichUniverseEligible",
    "PrimaryModelRoute", "PooledChallengerEligible",
    "SubmissionFile", "Pred", "RequestOrderInternal",
    "Team1SnapshotAvailable", "Team2SnapshotAvailable",
    "BothSnapshotsAvailable", "Team1SeedAvailable", "Team2SeedAvailable",
    "BothSeedsAvailable", "FeatureRoute", "PooledChallengerRoute",
}

MATCHUP_FEATURE_COLUMNS = [
    column
    for column in historical_matchups.columns
    if column not in TARGET_AND_ID_COLUMNS
    and column not in {"Gender", "Season", "Team1ID", "Team2ID"}
    and "max_source_season" not in column
    and (
        pd.api.types.is_numeric_dtype(historical_matchups[column])
        or pd.api.types.is_bool_dtype(historical_matchups[column])
    )
]

def feature_availability(column: str) -> str:
    return FEATURE_META.get(column, {}).get("Availability", "common")


def feature_universe(column: str) -> str:
    return FEATURE_META.get(column, {}).get("Universe", "compact")


def is_seed_feature(column: str) -> bool:
    return (
        "seed__" in column
        or column.startswith("seedprior__")
        or column.startswith("matchup__seed")
        or column in {
            "matchup__both_seeds_available",
            "matchup__team1_is_seed_favorite",
            "matchup__equal_seed",
            "matchup__team1_seed",
            "matchup__team2_seed",
        }
    )


def architecture_candidates(
    *,
    gender: str,
    universe: str,
    seed_aware: bool,
    pooled: bool = False,
) -> list[str]:
    allowed: list[str] = []
    for column in MATCHUP_FEATURE_COLUMNS:
        availability = feature_availability(column)
        feature_scope = feature_universe(column)

        if pooled and availability != "common":
            continue
        if gender == "W" and availability == "men_only":
            continue
        if gender == "M" and availability == "women_only":
            continue
        if universe == "compact" and feature_scope == "rich":
            continue
        if not seed_aware and is_seed_feature(column):
            continue
        allowed.append(column)

    if pooled and "context__women" not in allowed:
        allowed.append("context__women")
    return sorted(set(allowed))


CANDIDATE_SETS = {
    "men_compact_seed_free": architecture_candidates(
        gender="M", universe="compact", seed_aware=False
    ),
    "men_compact_seed_aware": architecture_candidates(
        gender="M", universe="compact", seed_aware=True
    ),
    "men_rich_seed_free": architecture_candidates(
        gender="M", universe="rich", seed_aware=False
    ),
    "men_rich_seed_aware": architecture_candidates(
        gender="M", universe="rich", seed_aware=True
    ),
    "women_compact_seed_free": architecture_candidates(
        gender="W", universe="compact", seed_aware=False
    ),
    "women_compact_seed_aware": architecture_candidates(
        gender="W", universe="compact", seed_aware=True
    ),
    "women_rich_seed_free": architecture_candidates(
        gender="W", universe="rich", seed_aware=False
    ),
    "women_rich_seed_aware": architecture_candidates(
        gender="W", universe="rich", seed_aware=True
    ),
    "pooled_rich_seed_free": architecture_candidates(
        gender="Pooled", universe="rich", seed_aware=False, pooled=True
    ),
    "pooled_rich_seed_aware": architecture_candidates(
        gender="Pooled", universe="rich", seed_aware=True, pooled=True
    ),
}

candidate_summary = pd.DataFrame(
    [
        {
            "CandidateSet": name,
            "Features": len(columns),
            "SeedAware": "seed_aware" in name,
            "RichUniverse": "rich" in name,
            "Pooled": name.startswith("pooled"),
        }
        for name, columns in CANDIDATE_SETS.items()
    ]
)
(FEATURE_REPORTS / "candidate_feature_sets.json").write_text(
    json.dumps(CANDIDATE_SETS, indent=2),
    encoding="utf-8",
)
candidate_summary.to_csv(
    FEATURE_REPORTS / "candidate_feature_set_summary.csv",
    index=False,
)

candidate_summary


## 15. Feature registry, source provenance, and leakage trap scan

The registry is the audit trail for every model candidate. Direct labels, scores, identifiers, same-
season tournament outcomes, and provenance-only fields are prohibited from candidate lists.


In [ ]:
all_present_columns = set(team_feature_store.columns).union(
    historical_matchups.columns
).union(stage2_matchups.columns)

feature_registry = pd.DataFrame(
    [
        metadata
        for name, metadata in FEATURE_META.items()
        if name in all_present_columns
    ]
).drop_duplicates("Feature")

feature_registry["PresentInTeamStore"] = feature_registry["Feature"].isin(
    team_feature_store.columns
)
feature_registry["PresentInHistoricalMatchups"] = feature_registry["Feature"].isin(
    historical_matchups.columns
)
feature_registry["PresentInStage2Matchups"] = feature_registry["Feature"].isin(
    stage2_matchups.columns
)
feature_registry["InAnyCandidateSet"] = feature_registry["Feature"].isin(
    set().union(*map(set, CANDIDATE_SETS.values()))
)
feature_registry["TargetDerivedPrior"] = feature_registry["Block"].isin(
    ["prior_program_history", "prior_coach_history", "prequential_seed_priors"]
)
feature_registry["RequiresFoldFittedPreprocessing"] = True
feature_registry = feature_registry.sort_values(
    ["Availability", "Universe", "Block", "Feature"]
).reset_index(drop=True)

BANNED_MODEL_COLUMNS = {
    "Team1Win", "Team1Margin", "Team1Score", "Team2Score",
    "WScore", "LScore", "WTeamID", "LTeamID", "GameKey",
    "TargetKey", "DayNum", "NumOT", "Team1Loc", "WLoc",
    "Team1ID", "Team2ID", "TeamID",
}

candidate_union = set().union(*map(set, CANDIDATE_SETS.values()))
leaked_exact = sorted(candidate_union.intersection(BANNED_MODEL_COLUMNS))
assert not leaked_exact, f"Direct leakage/identifier fields entered candidates: {leaked_exact}"

for name, columns in CANDIDATE_SETS.items():
    assert len(columns) == len(set(columns)), f"Duplicate candidates in {name}"
    assert not set(columns).intersection(BANNED_MODEL_COLUMNS)
    if "women" in name or "pooled" in name:
        men_only = [
            column for column in columns
            if feature_availability(column) == "men_only"
        ]
        assert not men_only, f"{name} contains men's-only fields: {men_only[:10]}"
    if "seed_free" in name:
        seed_columns = [column for column in columns if is_seed_feature(column)]
        assert not seed_columns, f"{name} contains seed fields: {seed_columns[:10]}"

# Provenance checks for target-derived team priors.
for provenance_column in (
    "prior__program_max_source_season",
    "coach__max_source_season",
):
    if provenance_column in team_feature_store.columns:
        valid = team_feature_store[provenance_column].notna()
        assert (
            team_feature_store.loc[valid, provenance_column]
            < team_feature_store.loc[valid, "Season"]
        ).all()

# No infinities are allowed in model-candidate matrices.
for name, frame in (
    ("historical", historical_matchups),
    ("stage2", stage2_matchups),
):
    columns = [column for column in candidate_union if column in frame.columns]
    numeric = frame[columns].select_dtypes(include=[np.number, "bool"])
    inf_count = int(np.isinf(numeric.to_numpy(dtype=float, na_value=np.nan)).sum())
    assert inf_count == 0, f"{name} feature store contains {inf_count} infinities."

feature_registry.to_csv(
    FEATURE_REPORTS / "feature_registry.csv",
    index=False,
)
print("Registered features:", len(feature_registry))
print("Candidate union:", len(candidate_union))
print("Direct leakage fields found:", leaked_exact)
feature_registry.groupby(
    ["Availability", "Universe", "Block"], observed=True
).size().to_frame("Features").reset_index()


## 16. Matchup symmetry audit

For difference-vector models, swapping Team1 and Team2 must negate directional features and preserve
absolute/context features. Seed priors must complement to one. This audit catches merge-direction,
location, and sign bugs before any classifier sees the data.


In [ ]:
development_sample = (
    target_registry.loc[target_registry["DatasetRole"].eq("development")]
    .sample(
        n=min(500, int(target_registry["DatasetRole"].eq("development").sum())),
        random_state=2026,
    )
    .reset_index(drop=True)
)
swapped_requests = development_sample.copy()
swapped_requests[["Team1ID", "Team2ID"]] = swapped_requests[
    ["Team2ID", "Team1ID"]
].to_numpy()
if "Team1Win" in swapped_requests.columns:
    swapped_requests["Team1Win"] = 1 - swapped_requests["Team1Win"]
if "Team1Margin" in swapped_requests.columns:
    swapped_requests["Team1Margin"] = -swapped_requests["Team1Margin"]

original_sample_features = build_matchup_store(
    development_sample, team_feature_store
)
swapped_sample_features = build_matchup_store(
    swapped_requests, team_feature_store
)

directional_columns = [
    column for column in original_sample_features.columns
    if column.startswith("diff__")
    or column in {
        "matchup__seed_diff",
        "interaction__adjusted_offense_defense_edge",
        "interaction__shooting_attack_defense_edge",
        "interaction__clean_shooting_attack_defense_edge",
        "interaction__turnover_pressure_edge",
        "interaction__rebounding_matchup_edge",
        "interaction__free_throw_pressure_edge",
        "interaction__strength_gap_squared_signed",
    }
]
symmetric_columns = [
    column for column in original_sample_features.columns
    if column.startswith("absdiff__")
    or column.startswith("mean__")
    or column in {
        "context__women",
        "context__season_centered_2000",
        "matchup__both_seeds_available",
        "matchup__seed_gap_abs",
        "matchup__equal_seed",
        "interaction__pace_gap",
        "interaction__pace_mean",
        "interaction__combined_margin_volatility",
    }
]

symmetry_records: list[dict[str, Any]] = []
for column in directional_columns:
    if column not in swapped_sample_features.columns:
        continue
    a = pd.to_numeric(original_sample_features[column], errors="coerce").to_numpy()
    b = pd.to_numeric(swapped_sample_features[column], errors="coerce").to_numpy()
    valid = np.isfinite(a) & np.isfinite(b)
    max_error = float(np.max(np.abs(a[valid] + b[valid]))) if valid.any() else 0.0
    symmetry_records.append(
        {
            "Feature": column,
            "ExpectedBehavior": "antisymmetric",
            "ComparableRows": int(valid.sum()),
            "MaxAbsoluteError": max_error,
            "Passed": max_error < 1e-5,
        }
    )

for column in symmetric_columns:
    if column not in swapped_sample_features.columns:
        continue
    a = pd.to_numeric(original_sample_features[column], errors="coerce").to_numpy()
    b = pd.to_numeric(swapped_sample_features[column], errors="coerce").to_numpy()
    valid = np.isfinite(a) & np.isfinite(b)
    max_error = float(np.max(np.abs(a[valid] - b[valid]))) if valid.any() else 0.0
    symmetry_records.append(
        {
            "Feature": column,
            "ExpectedBehavior": "symmetric",
            "ComparableRows": int(valid.sum()),
            "MaxAbsoluteError": max_error,
            "Passed": max_error < 1e-5,
        }
    )

# Explicitly test seed-prior complementarity on seeded historical games.
seeded_sample = historical_seed_prior_source.loc[
    historical_seed_prior_source["BothSeeds"]
    & ~historical_seed_prior_source["EqualSeeds"]
].sample(
    n=min(500, int(
        (
            historical_seed_prior_source["BothSeeds"]
            & ~historical_seed_prior_source["EqualSeeds"]
        ).sum()
    )),
    random_state=2026,
)
seeded_swapped = seeded_sample.copy()
seeded_swapped[["Team1ID", "Team2ID"]] = seeded_swapped[
    ["Team2ID", "Team1ID"]
].to_numpy()
seeded_swapped[["Team1Seed", "Team2Seed"]] = seeded_swapped[
    ["Team2Seed", "Team1Seed"]
].to_numpy()
seeded_swapped["Team1Win"] = 1 - seeded_swapped["Team1Win"]

prior_original = build_prequential_seed_priors(
    seeded_sample, seeded_targets
)
prior_swapped = build_prequential_seed_priors(
    seeded_swapped, seeded_targets
)
for pseudocount in FEATURE_CONFIG["seed_prior_pseudocounts"]:
    tag = str(pseudocount).replace(".", "p")
    column = f"seedprior__team1_prob_pc{tag}"
    error = np.abs(
        prior_original[column].to_numpy(dtype=float)
        + prior_swapped[column].to_numpy(dtype=float)
        - 1.0
    )
    max_error = float(np.nanmax(error)) if len(error) else 0.0
    symmetry_records.append(
        {
            "Feature": column,
            "ExpectedBehavior": "probability_complement",
            "ComparableRows": len(error),
            "MaxAbsoluteError": max_error,
            "Passed": max_error < 1e-10,
        }
    )

symmetry_audit = pd.DataFrame(symmetry_records)
symmetry_audit.to_csv(
    FEATURE_REPORTS / "matchup_symmetry_audit.csv",
    index=False,
)
failed_symmetry = symmetry_audit.loc[~symmetry_audit["Passed"]]
if not failed_symmetry.empty:
    display(failed_symmetry.head(30))
    raise AssertionError("One or more matchup symmetry checks failed.")

print("Symmetry checks passed:", len(symmetry_audit))
symmetry_audit.sort_values(
    "MaxAbsoluteError", ascending=False
).head(20)


## 17. Missingness, route coverage, and source-availability diagnostics

Missing values are information—not an invitation to globally impute before splitting. This section
measures them only. Notebook `03` will fit every imputer inside the relevant inner-training fold.


In [ ]:
def feature_missingness_report(
    frame: pd.DataFrame,
    feature_columns: Sequence[str],
    grouping_columns: Sequence[str],
    population_name: str,
) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    grouped = frame.groupby(list(grouping_columns), observed=True, dropna=False)
    for group_key, group in grouped:
        key_values = (
            group_key if isinstance(group_key, tuple) else (group_key,)
        )
        key_dict = dict(zip(grouping_columns, key_values, strict=True))
        for column in feature_columns:
            if column not in group.columns:
                continue
            series = group[column]
            records.append(
                {
                    "Population": population_name,
                    **key_dict,
                    "Feature": column,
                    "Block": FEATURE_META.get(column, {}).get("Block", "unregistered"),
                    "Availability": FEATURE_META.get(column, {}).get(
                        "Availability", "common"
                    ),
                    "Universe": FEATURE_META.get(column, {}).get(
                        "Universe", "compact"
                    ),
                    "Rows": len(group),
                    "NonMissing": int(series.notna().sum()),
                    "MissingRate": float(series.isna().mean()),
                    "UniqueNonMissing": int(series.nunique(dropna=True)),
                }
            )
    return pd.DataFrame(records)


historical_missingness = feature_missingness_report(
    historical_matchups,
    sorted(candidate_union),
    ["DatasetRole", "Gender", "FeatureRoute"],
    "historical_tournament_matchups",
)
stage2_missingness = feature_missingness_report(
    stage2_matchups,
    sorted(candidate_union),
    ["Gender", "FeatureRoute"],
    "stage2_matchups",
)
missingness_report = pd.concat(
    [historical_missingness, stage2_missingness],
    ignore_index=True,
)
missingness_report.to_csv(
    FEATURE_REPORTS / "matchup_feature_missingness.csv",
    index=False,
)

block_coverage = (
    missingness_report.groupby(
        [
            "Population", "DatasetRole" if "DatasetRole" in missingness_report.columns else "Population",
            "Gender", "FeatureRoute", "Block",
        ],
        observed=True,
        dropna=False,
    )
    .agg(
        Features=("Feature", "nunique"),
        MeanMissingRate=("MissingRate", "mean"),
        MedianMissingRate=("MissingRate", "median"),
        MaxMissingRate=("MissingRate", "max"),
    )
    .reset_index()
)
# Remove the duplicated helper column created when DatasetRole is absent.
block_coverage.to_csv(
    FEATURE_REPORTS / "feature_block_coverage.csv",
    index=False,
)

route_coverage = (
    stage2_matchups.groupby(["Gender", "FeatureRoute"], observed=True)
    .agg(
        Matchups=("ID", "nunique"),
        BothSnapshotsAvailable=("BothSnapshotsAvailable", "mean"),
        BothSeedsAvailable=("matchup__both_seeds_available", "mean"),
    )
    .reset_index()
)
route_coverage.to_csv(
    FEATURE_REPORTS / "stage2_feature_route_coverage.csv",
    index=False,
)

print("Missingness rows:", f"{len(missingness_report):,}")
print("Stage 2 route coverage:")
display(route_coverage)
missingness_report.sort_values(
    "MissingRate", ascending=False
).head(30)


## 18. Near-constant, redundancy, and multicollinearity diagnostics

Highly correlated candidates are not deleted here. They are reported so notebook `03` can compare
block-level and regularized selections inside nested folds. Tree models, elastic-net models, and
explanation methods react differently to redundant signals.


In [ ]:
diagnostic_feature_union = sorted(
    set(CANDIDATE_SETS["men_rich_seed_aware"])
    .union(CANDIDATE_SETS["women_rich_seed_aware"])
    .union(CANDIDATE_SETS["pooled_rich_seed_aware"])
)

near_constant_records: list[dict[str, Any]] = []
correlation_records: list[dict[str, Any]] = []
matrix_condition_records: list[dict[str, Any]] = []

development_rows = historical_matchups.loc[
    historical_matchups["DatasetRole"].eq("development")
].copy()

for gender in ("M", "W"):
    subset = development_rows.loc[development_rows["Gender"].eq(gender)]
    available = [
        column for column in diagnostic_feature_union
        if column in subset.columns
    ]
    usable: list[str] = []

    for column in available:
        series = pd.to_numeric(subset[column], errors="coerce")
        nonmissing = series.dropna()
        unique = int(nonmissing.nunique())
        variance = float(nonmissing.var()) if len(nonmissing) > 1 else 0.0
        missing_rate = float(series.isna().mean())
        near_constant_records.append(
            {
                "Gender": gender,
                "Feature": column,
                "Block": FEATURE_META.get(column, {}).get("Block", "unregistered"),
                "Rows": len(series),
                "MissingRate": missing_rate,
                "UniqueNonMissing": unique,
                "Variance": variance,
                "NearConstant": unique <= 1 or variance <= 1e-12,
            }
        )
        if unique > 1 and variance > 1e-12 and missing_rate <= 0.50:
            usable.append(column)

    if usable:
        matrix = subset[usable].apply(pd.to_numeric, errors="coerce")
        matrix = matrix.fillna(matrix.median(numeric_only=True))
        corr = matrix.corr().abs()
        upper = np.triu(np.ones(corr.shape, dtype=bool), k=1)
        rows, cols = np.where(
            upper
            & (
                corr.to_numpy()
                >= float(FEATURE_CONFIG["correlation_warning_threshold"])
            )
        )
        for row_idx, col_idx in zip(rows, cols, strict=True):
            correlation_records.append(
                {
                    "Gender": gender,
                    "FeatureA": usable[row_idx],
                    "FeatureB": usable[col_idx],
                    "AbsoluteCorrelation": float(corr.iat[row_idx, col_idx]),
                    "BlockA": FEATURE_META.get(usable[row_idx], {}).get(
                        "Block", "unregistered"
                    ),
                    "BlockB": FEATURE_META.get(usable[col_idx], {}).get(
                        "Block", "unregistered"
                    ),
                }
            )

        # Matrix diagnostics on a capped subset avoid turning diagnostics into a memory problem.
        rank_features = usable[: min(250, len(usable))]
        rank_matrix = matrix[rank_features].to_numpy(dtype=float)
        means = np.nanmean(rank_matrix, axis=0)
        stds = np.nanstd(rank_matrix, axis=0)
        stds[stds == 0] = 1.0
        rank_matrix = (rank_matrix - means) / stds
        singular_values = np.linalg.svd(
            rank_matrix,
            full_matrices=False,
            compute_uv=False,
        )
        positive = singular_values[singular_values > 1e-10]
        condition = (
            float(positive.max() / positive.min())
            if len(positive) else np.nan
        )
        weights = np.square(positive)
        weights = weights / weights.sum() if weights.sum() else weights
        effective_rank = (
            float(np.exp(-(weights * np.log(weights + 1e-300)).sum()))
            if len(weights) else 0.0
        )
        matrix_condition_records.append(
            {
                "Gender": gender,
                "Rows": len(subset),
                "FeaturesConsidered": len(rank_features),
                "NumericalRank": int(np.linalg.matrix_rank(rank_matrix)),
                "EffectiveRank": effective_rank,
                "ConditionNumber": condition,
            }
        )

near_constant_report = pd.DataFrame(near_constant_records)
correlation_report = pd.DataFrame(correlation_records)
matrix_condition_report = pd.DataFrame(matrix_condition_records)

near_constant_report.to_csv(
    FEATURE_REPORTS / "near_constant_features.csv",
    index=False,
)
correlation_report.to_csv(
    FEATURE_REPORTS / "high_correlation_pairs.csv",
    index=False,
)
matrix_condition_report.to_csv(
    FEATURE_REPORTS / "feature_matrix_condition.csv",
    index=False,
)

print("Near-constant feature findings:", int(near_constant_report["NearConstant"].sum()))
print(
    "High-correlation pairs:",
    len(correlation_report),
    f"(threshold >= {FEATURE_CONFIG['correlation_warning_threshold']})",
)
matrix_condition_report


## 19. Temporal drift diagnostics

Drift is measured against the development era separately by gender. The report compares development
to the locked 2022–2025 benchmark and to the **seed-aware** 2026 matchup population. It does not use
outcomes to select features.


In [ ]:
from scipy.stats import ks_2samp


def population_stability_index(
    reference: pd.Series,
    comparison: pd.Series,
    bins: int = 10,
) -> float:
    ref = pd.to_numeric(reference, errors="coerce").dropna().to_numpy(dtype=float)
    cur = pd.to_numeric(comparison, errors="coerce").dropna().to_numpy(dtype=float)
    if len(ref) < 20 or len(cur) < 20:
        return float("nan")
    edges = np.unique(
        np.quantile(ref, np.linspace(0.0, 1.0, bins + 1))
    )
    if len(edges) < 3:
        return 0.0
    edges[0] = -np.inf
    edges[-1] = np.inf
    ref_counts, _ = np.histogram(ref, bins=edges)
    cur_counts, _ = np.histogram(cur, bins=edges)
    ref_pct = np.clip(ref_counts / ref_counts.sum(), 1e-6, None)
    cur_pct = np.clip(cur_counts / cur_counts.sum(), 1e-6, None)
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))


def drift_record(
    feature: str,
    reference: pd.Series,
    comparison: pd.Series,
    *,
    gender: str,
    comparison_name: str,
) -> dict[str, Any]:
    ref = pd.to_numeric(reference, errors="coerce")
    cur = pd.to_numeric(comparison, errors="coerce")
    ref_nonmissing = ref.dropna()
    cur_nonmissing = cur.dropna()
    ref_std = float(ref_nonmissing.std()) if len(ref_nonmissing) > 1 else np.nan
    standardized_mean_difference = (
        float((cur_nonmissing.mean() - ref_nonmissing.mean()) / ref_std)
        if np.isfinite(ref_std) and ref_std > 0 and len(cur_nonmissing)
        else np.nan
    )
    ks = (
        float(ks_2samp(ref_nonmissing, cur_nonmissing).statistic)
        if len(ref_nonmissing) >= 20 and len(cur_nonmissing) >= 20
        else np.nan
    )
    return {
        "Gender": gender,
        "Comparison": comparison_name,
        "Feature": feature,
        "Block": FEATURE_META.get(feature, {}).get("Block", "unregistered"),
        "ReferenceRows": len(ref),
        "ComparisonRows": len(cur),
        "ReferenceMissingRate": float(ref.isna().mean()),
        "ComparisonMissingRate": float(cur.isna().mean()),
        "MissingRateDelta": float(cur.isna().mean() - ref.isna().mean()),
        "ReferenceMean": float(ref_nonmissing.mean()) if len(ref_nonmissing) else np.nan,
        "ComparisonMean": float(cur_nonmissing.mean()) if len(cur_nonmissing) else np.nan,
        "StandardizedMeanDifference": standardized_mean_difference,
        "KSStatistic": ks,
        "PSI": population_stability_index(ref, cur),
    }


drift_records: list[dict[str, Any]] = []
for gender, candidate_name in (
    ("M", "men_rich_seed_aware"),
    ("W", "women_rich_seed_aware"),
):
    reference = historical_matchups.loc[
        historical_matchups["DatasetRole"].eq("development")
        & historical_matchups["Gender"].eq(gender)
        & historical_matchups["FeatureRoute"].eq("seed_aware")
    ]
    locked = historical_matchups.loc[
        historical_matchups["DatasetRole"].eq("locked_benchmark")
        & historical_matchups["Gender"].eq(gender)
        & historical_matchups["FeatureRoute"].eq("seed_aware")
    ]
    target = stage2_matchups.loc[
        stage2_matchups["Gender"].eq(gender)
        & stage2_matchups["FeatureRoute"].eq("seed_aware")
    ]

    features = [
        column for column in CANDIDATE_SETS[candidate_name]
        if column in reference.columns
        and (
            column.startswith("diff__")
            or column.startswith("interaction__")
            or column.startswith("matchup__")
            or column.startswith("seedprior__team1_prob")
        )
    ]
    for feature in features:
        if reference[feature].notna().sum() < 20:
            continue
        drift_records.append(
            drift_record(
                feature,
                reference[feature],
                locked[feature],
                gender=gender,
                comparison_name="locked_2022_2025",
            )
        )
        drift_records.append(
            drift_record(
                feature,
                reference[feature],
                target[feature],
                gender=gender,
                comparison_name="stage2_2026_seed_aware",
            )
        )

DRIFT_COLUMNS = [
    "Gender", "Comparison", "Feature", "Block", "ReferenceRows",
    "ComparisonRows", "ReferenceMissingRate", "ComparisonMissingRate",
    "MissingRateDelta", "ReferenceMean", "ComparisonMean",
    "StandardizedMeanDifference", "KSStatistic", "PSI",
]
drift_report = pd.DataFrame(drift_records, columns=DRIFT_COLUMNS)
drift_report["PSIWarning"] = (
    drift_report["PSI"].ge(
        float(FEATURE_CONFIG["drift_psi_warning_threshold"])
    )
    if not drift_report.empty
    else pd.Series(dtype=bool)
)
drift_report.to_csv(
    FEATURE_REPORTS / "temporal_feature_drift.csv",
    index=False,
)

if drift_report.empty:
    drift_block_summary = pd.DataFrame(
        columns=[
            "Gender", "Comparison", "Block", "Features", "MedianPSI",
            "MaxPSI", "PSIWarnings", "MedianAbsSMD", "MaxKS",
        ]
    )
else:
    drift_block_summary = (
        drift_report.groupby(
            ["Gender", "Comparison", "Block"], observed=True
        )
        .agg(
            Features=("Feature", "nunique"),
            MedianPSI=("PSI", "median"),
            MaxPSI=("PSI", "max"),
            PSIWarnings=("PSIWarning", "sum"),
            MedianAbsSMD=(
                "StandardizedMeanDifference",
                lambda s: float(s.abs().median()),
            ),
            MaxKS=("KSStatistic", "max"),
        )
        .reset_index()
    )
drift_block_summary.to_csv(
    FEATURE_REPORTS / "temporal_drift_by_block.csv",
    index=False,
)

print("Drift feature comparisons:", len(drift_report))
print("PSI warnings:", int(drift_report["PSIWarning"].sum()))
drift_report.sort_values("PSI", ascending=False).head(30)


## 20. Rating concordance and feature-block visualization

Independent ratings should agree broadly without being identical. Perfect correlation means wasted
complexity; weak correlation can signal either useful diversity or a broken method. These plots are
diagnostics only and do not select a winner.


In [ ]:
core_rating_columns = [
    column for column in [
        "rating__elo_standard",
        "rating__elo_mov_log",
        "rating__elo_mov_538",
        "rating__margin_ridge_a5p0",
        "rating__margin_ridge_a25p0",
        "rating__bt_c0p25",
        "rating__bt_c1p0",
        "rating__colley",
        "rating__pagerank_binary",
        "rating__pagerank_margin",
        "adj__net_rtg_a10p0",
        "adj__net_rtg_a50p0",
        "massey__strength_mean",
    ]
    if column in team_feature_store.columns
]

rating_concordance_records: list[pd.DataFrame] = []
for gender in ("M", "W"):
    subset = team_feature_store.loc[
        team_feature_store["Gender"].eq(gender)
        & team_feature_store["Season"].le(DEVELOPMENT_LAST_SEASON),
        core_rating_columns,
    ]
    usable = [
        column for column in core_rating_columns
        if subset[column].notna().sum() >= 50
        and subset[column].nunique(dropna=True) > 1
    ]
    if not usable:
        continue
    corr = subset[usable].corr(method="spearman")
    long = (
        corr.rename_axis("RatingA")
        .reset_index()
        .melt(id_vars="RatingA", var_name="RatingB", value_name="SpearmanCorrelation")
    )
    long["Gender"] = gender
    rating_concordance_records.append(long)

    fig, ax = plt.subplots(figsize=(max(8, len(usable) * 0.65), max(7, len(usable) * 0.58)))
    image = ax.imshow(corr.to_numpy(), vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(np.arange(len(usable)), labels=usable, rotation=90)
    ax.set_yticks(np.arange(len(usable)), labels=usable)
    ax.set_title(f"{gender}: development-era rating concordance (Spearman)")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / f"rating_concordance_{gender}.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()

rating_concordance = (
    pd.concat(rating_concordance_records, ignore_index=True)
    if rating_concordance_records
    else pd.DataFrame()
)
rating_concordance.to_csv(
    FEATURE_REPORTS / "rating_concordance.csv",
    index=False,
)

block_counts = (
    feature_registry.loc[feature_registry["InAnyCandidateSet"]]
    .groupby(["Block", "Availability"], observed=True)
    .size()
    .unstack(fill_value=0)
    .sort_values(by=list(
        feature_registry["Availability"].dropna().unique()
    ), ascending=False)
)
fig, ax = plt.subplots(figsize=(12, 7))
block_counts.plot(kind="barh", stacked=True, ax=ax)
ax.set_title("Model-candidate features by block and source availability")
ax.set_xlabel("Feature count")
ax.set_ylabel("Feature block")
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "feature_counts_by_block.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

block_counts


## 21. Pre-register the feature-block ablation ladder

Notebook `03` must not search arbitrary subsets until something looks good. It will evaluate this
ordered ladder and targeted leave-one-block-out tests inside nested folds, separately for men, women,
and the pooled challenger.


In [ ]:
FEATURE_BLOCK_ABLATION_PLAN = pd.DataFrame(
    [
        {
            "Stage": 0,
            "Experiment": "constant_probability",
            "Blocks": "none",
            "Purpose": "absolute Brier baseline",
        },
        {
            "Stage": 1,
            "Experiment": "seed_only",
            "Blocks": "selection_committee_prior + prequential_seed_priors",
            "Purpose": "committee-information baseline",
        },
        {
            "Stage": 2,
            "Experiment": "compact_basic",
            "Blocks": "compact_performance",
            "Purpose": "long-history results baseline",
        },
        {
            "Stage": 3,
            "Experiment": "compact_plus_dynamic_ratings",
            "Blocks": "compact_performance + dynamic_ratings + global_strength_ratings",
            "Purpose": "chronological and global strength",
        },
        {
            "Stage": 4,
            "Experiment": "schedule_context",
            "Blocks": "previous + schedule_and_accomplishment + conference_context",
            "Purpose": "opponent and accomplishment adjustment",
        },
        {
            "Stage": 5,
            "Experiment": "rich_efficiency",
            "Blocks": "previous + detailed_efficiency + opponent_adjusted_efficiency",
            "Purpose": "possession and style information",
        },
        {
            "Stage": 6,
            "Experiment": "history_context",
            "Blocks": "previous + prior_program_history + prior_coach_history",
            "Purpose": "strictly prior tournament experience",
        },
        {
            "Stage": 7,
            "Experiment": "men_massey",
            "Blocks": "previous + massey_consensus (men only)",
            "Purpose": "public ranking consensus and disagreement",
        },
        {
            "Stage": 8,
            "Experiment": "matchup_interactions",
            "Blocks": "previous + matchup_interactions",
            "Purpose": "basketball style compatibility",
        },
        {
            "Stage": 9,
            "Experiment": "full_candidate_bank",
            "Blocks": "all legally available blocks",
            "Purpose": "regularized/tree selection under nested validation",
        },
        {
            "Stage": 10,
            "Experiment": "leave_one_block_out",
            "Blocks": "full bank minus one block at a time",
            "Purpose": "marginal contribution and robustness",
        },
        {
            "Stage": 11,
            "Experiment": "quality_sensitivity",
            "Blocks": "all vs clean-possession vs robust variants",
            "Purpose": "data-quality dependence",
        },
        {
            "Stage": 12,
            "Experiment": "architecture_comparison",
            "Blocks": "separate gender vs pooled common vs partial pooling",
            "Purpose": "scientific architecture decision",
        },
    ]
)
FEATURE_BLOCK_ABLATION_PLAN.to_csv(
    FEATURE_REPORTS / "feature_block_ablation_plan.csv",
    index=False,
)
FEATURE_BLOCK_ABLATION_PLAN


## 22. Final leakage tests, artifact writing, and reproducibility manifest


In [ ]:
feature_checks: list[dict[str, Any]] = []


def add_feature_check(name: str, passed: bool, details: str) -> None:
    feature_checks.append(
        {"Check": name, "Passed": bool(passed), "Details": details}
    )


add_feature_check(
    "source split contract unchanged",
    FEATURE_CONFIG["source_split_contract_sha256"] == SPLITS["contract_sha256"],
    SPLITS["contract_sha256"],
)
add_feature_check(
    "team feature key unique",
    team_feature_store[KEY_COLUMNS].duplicated().sum() == 0,
    f"rows={len(team_feature_store):,}",
)
add_feature_check(
    "historical matchup key unique",
    historical_matchups["TargetKey"].is_unique,
    f"rows={len(historical_matchups):,}",
)
add_feature_check(
    "Stage 2 ID unique",
    stage2_matchups["ID"].is_unique,
    f"rows={len(stage2_matchups):,}",
)
add_feature_check(
    "historical lower-ID orientation",
    historical_matchups["Team1ID"].lt(historical_matchups["Team2ID"]).all(),
    "Team1ID < Team2ID",
)
add_feature_check(
    "Stage 2 lower-ID orientation",
    stage2_matchups["Team1ID"].lt(stage2_matchups["Team2ID"]).all(),
    "Team1ID < Team2ID",
)
add_feature_check(
    "Stage 2 full snapshot coverage",
    stage2_matchups["BothSnapshotsAvailable"].astype(bool).all(),
    f"coverage={stage2_matchups['BothSnapshotsAvailable'].mean():.6f}",
)
add_feature_check(
    "same-season cutoff preserved",
    int(base_snapshot["SnapshotDayNum"].max()) <= CUTOFF_DAY,
    f"max snapshot day={base_snapshot['SnapshotDayNum'].max()}",
)
add_feature_check(
    "no target or identifier fields in candidates",
    not bool(candidate_union.intersection(BANNED_MODEL_COLUMNS)),
    f"candidate_features={len(candidate_union):,}",
)
add_feature_check(
    "women and pooled candidates exclude men-only sources",
    all(
        feature_availability(column) != "men_only"
        for name, columns in CANDIDATE_SETS.items()
        if name.startswith(("women", "pooled"))
        for column in columns
    ),
    "Massey/coach features restricted to men's sets",
)
add_feature_check(
    "seed-free routes exclude seed features",
    all(
        not is_seed_feature(column)
        for name, columns in CANDIDATE_SETS.items()
        if "seed_free" in name
        for column in columns
    ),
    "all seed-free candidate sets",
)
add_feature_check(
    "historical seed priors are prequential",
    (
        historical_matchups.loc[
            historical_matchups["seedprior__max_source_season"].notna(),
            "seedprior__max_source_season",
        ]
        < historical_matchups.loc[
            historical_matchups["seedprior__max_source_season"].notna(),
            "Season",
        ]
    ).all(),
    "max source season < prediction season",
)
add_feature_check(
    "program priors use earlier seasons",
    (
        team_feature_store.loc[
            team_feature_store["prior__program_max_source_season"].notna(),
            "prior__program_max_source_season",
        ]
        < team_feature_store.loc[
            team_feature_store["prior__program_max_source_season"].notna(),
            "Season",
        ]
    ).all(),
    "max source season < prediction season",
)
add_feature_check(
    "all symmetry checks pass",
    symmetry_audit["Passed"].all(),
    f"checks={len(symmetry_audit):,}",
)
add_feature_check(
    "historical rows match target registry",
    len(historical_matchups) == len(target_registry),
    f"{len(historical_matchups):,} rows",
)
add_feature_check(
    "Stage 2 rows match routing registry",
    len(stage2_matchups) == len(submission_routing),
    f"{len(stage2_matchups):,} rows",
)
add_feature_check(
    "all candidate sets nonempty",
    all(len(columns) > 0 for columns in CANDIDATE_SETS.values()),
    json.dumps({name: len(columns) for name, columns in CANDIDATE_SETS.items()}),
)

feature_checks_df = pd.DataFrame(feature_checks)
failed_checks = feature_checks_df.loc[~feature_checks_df["Passed"]]
feature_checks_df.to_csv(
    FEATURE_REPORTS / "feature_store_leakage_and_integrity_checks.csv",
    index=False,
)
if not failed_checks.empty:
    display(failed_checks)
    raise AssertionError("Blocking feature-store checks failed.")

print("All blocking feature-store checks passed:", len(feature_checks_df))
feature_checks_df


In [ ]:
# Reduce artifact size without changing keys or labels.
def downcast_feature_frame(
    frame: pd.DataFrame,
    protected_columns: set[str],
) -> pd.DataFrame:
    result = frame.copy()
    for column in result.columns:
        if column in protected_columns:
            continue
        if result[column].dtype == "float64":
            result[column] = result[column].astype("float32")
        elif result[column].dtype == "int64":
            numeric = result[column]
            if numeric.notna().all():
                minimum = numeric.min()
                maximum = numeric.max()
                if np.iinfo(np.int16).min <= minimum <= maximum <= np.iinfo(np.int16).max:
                    result[column] = numeric.astype("int16")
                elif np.iinfo(np.int32).min <= minimum <= maximum <= np.iinfo(np.int32).max:
                    result[column] = numeric.astype("int32")
    return result


team_feature_store_out = downcast_feature_frame(
    team_feature_store,
    protected_columns={"Season", "TeamID", "TeamName", "ConfAbbrev", "TeamFeatureKey"},
)
historical_matchups_out = downcast_feature_frame(
    historical_matchups,
    protected_columns={
        "Season", "Team1ID", "Team2ID", "TargetKey", "GameKey",
        "DatasetRole", "Gender", "FeatureRoute",
    },
)
stage2_matchups_out = downcast_feature_frame(
    stage2_matchups,
    protected_columns={
        "Season", "Team1ID", "Team2ID", "ID", "SubmissionFile",
        "Gender", "FeatureRoute", "PrimaryModelRoute", "PooledChallengerRoute",
    },
)

output_paths = {
    "team_feature_store_v1": PROCESSED / "team_feature_store_v1.parquet",
    "historical_matchup_feature_store_v1": (
        PROCESSED / "historical_matchup_feature_store_v1.parquet"
    ),
    "stage2_matchup_feature_store_v1": (
        PROCESSED / "stage2_matchup_feature_store_v1.parquet"
    ),
}
team_feature_store_out.to_parquet(
    output_paths["team_feature_store_v1"],
    index=False,
    compression="zstd",
)
historical_matchups_out.to_parquet(
    output_paths["historical_matchup_feature_store_v1"],
    index=False,
    compression="zstd",
)
stage2_matchups_out.to_parquet(
    output_paths["stage2_matchup_feature_store_v1"],
    index=False,
    compression="zstd",
)

feature_registry.to_csv(
    FEATURE_REPORTS / "feature_registry_v1.csv",
    index=False,
)
artifact_hashes = {
    name: sha256_file(path)
    for name, path in output_paths.items()
}
artifact_manifest = pd.DataFrame(
    [
        {
            "Artifact": name,
            "Path": str(path),
            "Rows": len(
                team_feature_store_out
                if name == "team_feature_store_v1"
                else historical_matchups_out
                if name == "historical_matchup_feature_store_v1"
                else stage2_matchups_out
            ),
            "Columns": (
                team_feature_store_out.shape[1]
                if name == "team_feature_store_v1"
                else historical_matchups_out.shape[1]
                if name == "historical_matchup_feature_store_v1"
                else stage2_matchups_out.shape[1]
            ),
            "SizeMB": round(path.stat().st_size / 1024**2, 3),
            "SHA256": artifact_hashes[name],
            "SplitContractSHA256": SPLITS["contract_sha256"],
            "FeatureContractSHA256": FEATURE_CONFIG["feature_contract_sha256"],
        }
        for name, path in output_paths.items()
    ]
)
artifact_manifest.to_csv(
    FEATURE_REPORTS / "feature_store_artifact_manifest.csv",
    index=False,
)
artifact_manifest


In [ ]:
readiness = {
    "notebook": "02_feature_store_and_diagnostics.ipynb",
    "status": "complete",
    "split_contract_sha256": SPLITS["contract_sha256"],
    "feature_contract_sha256": FEATURE_CONFIG["feature_contract_sha256"],
    "target_season": TARGET_SEASON,
    "feature_cutoff_day": CUTOFF_DAY,
    "team_feature_rows": int(len(team_feature_store_out)),
    "team_feature_columns": int(team_feature_store_out.shape[1]),
    "registered_features": int(len(feature_registry)),
    "candidate_feature_union": int(len(candidate_union)),
    "candidate_sets": {
        name: len(columns) for name, columns in CANDIDATE_SETS.items()
    },
    "historical_matchup_rows": int(len(historical_matchups_out)),
    "historical_matchup_columns": int(historical_matchups_out.shape[1]),
    "stage2_matchup_rows": int(len(stage2_matchups_out)),
    "stage2_matchup_columns": int(stage2_matchups_out.shape[1]),
    "men_stage2_rows": int(stage2_matchups_out["Gender"].eq("M").sum()),
    "women_stage2_rows": int(stage2_matchups_out["Gender"].eq("W").sum()),
    "stage2_seed_aware_rows": int(
        stage2_matchups_out["FeatureRoute"].eq("seed_aware").sum()
    ),
    "stage2_seed_free_rows": int(
        stage2_matchups_out["FeatureRoute"].eq("seed_free_fallback").sum()
    ),
    "stable_massey_systems": stable_massey_systems,
    "symmetry_checks": int(len(symmetry_audit)),
    "symmetry_failures": int((~symmetry_audit["Passed"]).sum()),
    "blocking_feature_checks": int(len(feature_checks_df)),
    "blocking_feature_check_failures": int((~feature_checks_df["Passed"]).sum()),
    "near_constant_findings": int(
        near_constant_report["NearConstant"].sum()
    ),
    "high_correlation_pairs": int(len(correlation_report)),
    "drift_psi_warnings": int(drift_report["PSIWarning"].sum()),
    "artifact_sha256": artifact_hashes,
}
(FEATURE_REPORTS / "02_readiness_summary.json").write_text(
    json.dumps(readiness, indent=2),
    encoding="utf-8",
)

protocol = f'''# research-grade Feature Store Protocol

## Boundaries

- Split contract: `{SPLITS["contract_sha256"]}`
- Feature contract: `{FEATURE_CONFIG["feature_contract_sha256"]}`
- Target season: {TARGET_SEASON}
- Feature cutoff: DayNum {CUTOFF_DAY}
- Same-season NCAA outcomes are prohibited from team features.
- Program, coach, and seed priors use tournament outcomes from strictly earlier seasons only.

## Stores

- `data/processed/team_feature_store_v1.parquet`
- `data/processed/historical_matchup_feature_store_v1.parquet`
- `data/processed/stage2_matchup_feature_store_v1.parquet`

## Architecture support

- Men's separate model: common + men's-only features
- Women's separate model: common features without fake Massey values
- Pooled challenger: common features + explicit gender context
- Partial pooling: prediction-level blend selected from nested OOF only
- Seed-aware and seed-free routes have distinct candidate manifests

## Feature governance

- No feature is selected in notebook 02.
- Imputation, scaling, feature selection, hyperparameter tuning, calibration, and blending
  must be fit inside inner-training folds in notebook 03.
- Locked 2022–2025 outcomes cannot influence the recipe.
- Full, clean-possession, robust, recent, and EWM variants remain separate candidates.
- Correlation and drift findings are diagnostics, not automatic deletion rules.

## Next stage

Notebook 03 must generate nested out-of-fold predictions for the complete model ladder:
constant, seed logistic, corrected Elo+seed logistic, elastic-net logistic, XGBoost,
LightGBM, direct classification, point-margin regression, calibrated models, and
constrained gender-specific/pooled ensembles.
'''
(FEATURE_REPORTS / "FEATURE_STORE_PROTOCOL.md").write_text(
    protocol,
    encoding="utf-8",
)

print(json.dumps(readiness, indent=2))
assert readiness["status"] == "complete"
assert readiness["symmetry_failures"] == 0
assert readiness["blocking_feature_check_failures"] == 0

print(
    "\nNOTEBOOK 02 COMPLETE — leakage-safe research-grade candidate feature stores are ready."
)
print(
    "No tournament model, calibrator, feature selector, or ensemble weight has been chosen."
)
print(
    "Next: notebook 03 — nested out-of-fold model ladder, ablation, calibration, and ensemble laboratory."
)


# Stop here

Return the fully executed notebook and `reports/feature_engineering/02_readiness_summary.json` for
review. The next notebook will be:

```text
03_nested_oof_model_laboratory.ipynb
```

It will use the frozen outer/inner manifests to train and compare:

- constant, seed-only, corrected Elo+seed, and rich elastic-net baselines;
- XGBoost and LightGBM classifiers;
- XGBoost and LightGBM point-margin regressors;
- optional regularized neural challengers;
- Platt, temperature, isotonic, and spline-style calibration;
- separate men/women, pooled, and partial-pooling architectures;
- constrained nonnegative ensemble weights;
- feature-block ablations, season-clustered uncertainty, calibration curves, SHAP, and error analysis.

Nothing in notebook `03` may alter the split contract or select a recipe from the locked benchmark.
